In [6]:
import os, glob

# Find the brca-preprocessed dataset path
files = glob.glob('/kaggle/input/**/*cohort*', recursive=True)
print('cohort file found at:')
for f in files:
    print(f)

cohort file found at:
/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset/cohort_patient_ids (1).csv


In [14]:
OUT = '/kaggle/working'

In [ ]:
import os
root = '/kaggle/input/datasets'
for owner in os.listdir(root):
    for d in os.listdir(f'{root}/{owner}'):
        if 'histo' in d.lower():
            p = f'{root}/{owner}/{d}'
            print(p, '->', os.listdir(p)[:5])

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import timm
import cv2
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut, cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, matthews_corrcoef
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, ClassifierMixin
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
CLEAN_TILES = '/kaggle/input/datasets/mdafsar07/final-histo/Final_Clean_Tiles'   # <- CHANGED
BRCA_BASE   = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'
OUT         = '/kaggle/working'
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

LABEL_COLS  = ['label_cancer_stage','label_node_positive',
               'label_survival','label_recurrence','label_treatment_response']

LEAK_COLS = [
    'fu.follow_ups.days_to_recurrence','extra_days_to_recurrence',
    'extra_days_to_progression','demographic.days_to_death',
    'diagnoses.days_to_last_follow_up','fu.follow_ups.days_to_follow_up',
    'disease_response_wt-with tumor','disease_response_tf-tumor free',
    'disease_response_unknown','disease_response_wt-with tumor.1',
    'disease_response_tf-tumor free.1','disease_response_unknown.1',
    'fu.follow_ups.disease_response','extra_disease_response_tf-tumor free',
    'extra_disease_response_wt-with tumor','extra_disease_response_unknown',
    'progression_or_recurrence_yes','progression_or_recurrence_unknown',
    'ord_ajcc_pathologic_stage','ord_ajcc_pathologic_n',
    'ord_ajcc_pathologic_m','ord_ajcc_pathologic_t','ord_tumor_grade',
    'progression_or_recurrence_type_distant',
    'progression_or_recurrence_type_locoregional',
    'progression_or_recurrence_type_unknown',
]

def clean_X(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object or df[col].dtype == bool:
            df[col] = df[col].map(
                {'True':1,'False':0,True:1,False:0,
                 'true':1,'false':0,'yes':1,'no':0}
            ).fillna(df[col])
            df[col] = pd.to_numeric(df[col], errors='coerce')
    all_nan = df.columns[df.isna().all()].tolist()
    if all_nan: df = df.drop(columns=all_nan)
    imp = SimpleImputer(strategy='median')
    return pd.DataFrame(imp.fit_transform(df),
                        index=df.index, columns=df.columns)

def remove_leaks(df):
    return df.drop(columns=[c for c in LEAK_COLS if c in df.columns])

print('Loading clinical+genomic...')
X_clin_raw = pd.read_csv(f'{BRCA_BASE}/X_clinical.csv', index_col='patient_id')
X_clin     = clean_X(remove_leaks(X_clin_raw))
X_pca      = clean_X(remove_leaks(
    pd.read_csv(f'{BRCA_BASE}/X_pca_v2.csv', index_col='patient_id')))
X_genomic  = clean_X(X_pca.filter(like='PC'))
y_old      = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')

# Build 5-task labels
stage   = X_clin_raw['ord_ajcc_pathologic_stage']
n_stage = X_clin_raw['ord_ajcc_pathologic_n']
label_stage = pd.Series(np.nan, index=X_clin_raw.index)
label_stage[stage >= 8] = 1
label_stage[(stage >= 1) & (stage <= 6)] = 0
label_node = pd.Series(np.nan, index=X_clin_raw.index)
label_node[n_stage >= 6] = 1
label_node[(n_stage >= 2) & (n_stage <= 5)] = 0
y_full = pd.DataFrame({
    'label_cancer_stage'      : label_stage,
    'label_node_positive'     : label_node,
    'label_survival'          : y_old['label_survival'],
    'label_recurrence'        : y_old['label_recurrence'],
    'label_treatment_response': y_old['label_treatment_response'],
}, index=X_clin_raw.index)
y_full.to_csv(f'{OUT}/y_labels_4tasks.csv')
print(f'Clinical: {X_clin.shape}  Genomic: {X_genomic.shape}  Labels: {y_full.shape}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
resnet = resnet.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

def get_embeddings_batch(paths, batch_size=32):
    all_embs = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        tensors = []
        for p in batch:
            try: tensors.append(transform(Image.open(p).convert('RGB')))
            except: continue
        if not tensors: continue
        t = torch.stack(tensors).to(device)
        with torch.no_grad():
            embs = resnet(t).squeeze(-1).squeeze(-1).cpu().numpy()
        all_embs.append(embs)
    return np.vstack(all_embs) if all_embs else np.array([])

# Build manifest — max 150 tiles per patient
records = []
for slide in os.listdir(CLEAN_TILES):
    slide_dir = f'{CLEAN_TILES}/{slide}'
    if not os.path.isdir(slide_dir): continue
    for tile in os.listdir(slide_dir):
        if tile.endswith('.png'):
            records.append({
                'patient_id': '-'.join(slide.split('-')[:3]),
                'tile_path' : f'{slide_dir}/{tile}'
            })

manifest = pd.DataFrame(records)
manifest = (manifest.groupby('patient_id')
                    .apply(lambda x: x.sample(min(len(x), 150), random_state=42))
                    .reset_index(drop=True))
print(f'Patients: {manifest["patient_id"].nunique()}  Tiles: {len(manifest):,}')

patient_embs = {}
for pid in tqdm(manifest['patient_id'].unique(), desc='Extracting'):
    tiles = manifest[manifest['patient_id']==pid]['tile_path'].tolist()
    embs  = get_embeddings_batch(tiles)
    if len(embs) > 0:
        patient_embs[pid] = embs.mean(axis=0)

# PCA: 2048 → 50 PCs
histo_raw = pd.DataFrame(patient_embs).T
histo_raw.index.name = 'patient_id'
N_PCS = min(50, len(histo_raw)-1)
scaler_h = StandardScaler()
pca_h    = PCA(n_components=N_PCS, random_state=42)
histo_pca = pd.DataFrame(
    pca_h.fit_transform(scaler_h.fit_transform(histo_raw)),
    index=histo_raw.index,
    columns=[f'histo_PC{i+1}' for i in range(N_PCS)]
)
print(f'Histo PCA: {histo_pca.shape}  Var: {pca_h.explained_variance_ratio_.sum():.1%}')
histo_pca.to_csv(f'{OUT}/X_histopath_64.csv')

# Build 64-patient subsets
idx_64       = list(histo_pca.index)
X_clin_64    = X_clin.loc[idx_64]
X_genomic_64 = X_genomic.loc[idx_64]
X_pca_64     = X_pca.loc[idx_64]
y_64         = y_full.loc[idx_64]
X_multi_64   = clean_X(X_pca_64.join(histo_pca, how='inner'))

print(f'\nAll 64-patient matrices ready:')
print(f'  X_clin_64    : {X_clin_64.shape}')
print(f'  X_genomic_64 : {X_genomic_64.shape}')
print(f'  X_pca_64     : {X_pca_64.shape}')
print(f'  histo_pca    : {histo_pca.shape}')
print(f'  X_multi_64   : {X_multi_64.shape}')

LABEL_COLS = ['label_cancer_stage','label_node_positive',
              'label_survival','label_recurrence','label_treatment_response']
print(f'\nLabel distributions (64 patients):')
for col in LABEL_COLS:
    n = y_64[col].notna().sum()
    p = int(y_64[col].fillna(0).sum())
    print(f'  {col:<38} n={n}  pos={p}  rate={p/n:.1%}')

In [ ]:
def evaluate(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    try:    auc = roc_auc_score(y_true, y_prob)
    except: auc = 0.5
    return {
        'AUC'   : round(auc, 3),
        'F1'    : round(f1_score(y_true, y_pred, zero_division=0), 3),
        'BalAcc': round(balanced_accuracy_score(y_true, y_pred), 3),
        'MCC'   : round(matthews_corrcoef(y_true, y_pred), 3)
    }

def run_cv(X, y_series, model, use_loo=True):
    y = y_series.reindex(X.index).dropna().astype(int)
    X_v = X.loc[y.index]
    if y.nunique() < 2 or y.sum() < 3: return None
    k = min(3, int(y.sum()) - 1)
    pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=42, k_neighbors=k)),
        ('scaler', StandardScaler()),
        ('clf',    model)
    ])
    cv = LeaveOneOut() if use_loo else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    try:
        y_prob = cross_val_predict(pipe, X_v, y, cv=cv, method='predict_proba')[:, 1]
        return evaluate(y, y_prob)
    except Exception as e:
        print(f'    Error: {e}')
        return None

# LateFusion model
class LateFusion(BaseEstimator, ClassifierMixin):
    def __init__(self, n_clin, n_gen, random_state=42):
        self.n_clin = n_clin
        self.n_gen  = n_gen
        self.random_state = random_state
        self.classes_ = np.array([0, 1])

    def _split(self, X):
        return (X[:, :self.n_clin],
                X[:, self.n_clin:self.n_clin+self.n_gen],
                X[:, self.n_clin+self.n_gen:])

    def _expert(self, seed, n_components=10):
        return ImbPipeline([
            ('smote',  SMOTE(random_state=seed, k_neighbors=2)),
            ('scaler', StandardScaler()),
            ('pca',    PCA(n_components=n_components, random_state=seed)),
            ('clf',    LogisticRegression(
                           penalty='elasticnet', solver='saga',
                           l1_ratio=0.5, C=0.3, class_weight='balanced',
                           max_iter=2000, random_state=seed))
        ])

    def fit(self, X, y):
        X_c, X_g, X_h = self._split(X)
        nc = min(10, X_c.shape[1]-1, int(y.sum())-1)
        ng = min(10, X_g.shape[1]-1, int(y.sum())-1)
        nh = min(10, X_h.shape[1]-1, int(y.sum())-1)
        self.e_c_ = self._expert(self.random_state,   nc)
        self.e_g_ = self._expert(self.random_state+1, ng)
        self.e_h_ = self._expert(self.random_state+2, nh)
        self.e_c_.fit(X_c, y)
        self.e_g_.fit(X_g, y)
        self.e_h_.fit(X_h, y)
        p = np.column_stack([
            self.e_c_.predict_proba(X_c)[:, 1],
            self.e_g_.predict_proba(X_g)[:, 1],
            self.e_h_.predict_proba(X_h)[:, 1],
        ])
        self.meta_ = LogisticRegression(C=1.0, class_weight='balanced',
                                         max_iter=1000, random_state=self.random_state)
        self.meta_.fit(p, y)
        return self

    def predict_proba(self, X):
        X_c, X_g, X_h = self._split(X)
        p = np.column_stack([
            self.e_c_.predict_proba(X_c)[:, 1],
            self.e_g_.predict_proba(X_g)[:, 1],
            self.e_h_.predict_proba(X_h)[:, 1],
        ])
        return self.meta_.predict_proba(p)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

def make_models():
    return {
        'SVM'      : SVC(kernel='rbf', C=1.0, gamma='scale',
                         class_weight='balanced', probability=True, random_state=42),
        'XGBoost'  : xgb.XGBClassifier(n_estimators=100, max_depth=2,
                         learning_rate=0.05, subsample=0.8, min_child_weight=3,
                         scale_pos_weight=5, eval_metric='logloss', random_state=42),
        'LightGBM' : lgb.LGBMClassifier(n_estimators=100, max_depth=2,
                         learning_rate=0.05, subsample=0.8,
                         class_weight='balanced', random_state=42, verbose=-1),
        'ElasticNet': LogisticRegression(penalty='elasticnet', solver='saga',
                         l1_ratio=0.5, C=0.1, class_weight='balanced',
                         max_iter=2000, random_state=42),
    }

print('Models and functions defined!')

In [ ]:
# Build all experiment matrices
EXPERIMENTS = [
    ('1. Clinical only',     X_clin_64,                                    y_64, True),
    ('2. Genomic only',      X_genomic_64,                                 y_64, True),
    ('3. Histo only',        histo_pca,                                    y_64, True),
    ('4. Clinical+Genomic',  X_pca_64,                                     y_64, True),
    ('5. Clinical+Histo',    clean_X(X_clin_64.join(histo_pca)),           y_64, True),
    ('6. Genomic+Histo',     clean_X(X_genomic_64.join(histo_pca)),        y_64, True),
    ('7. All 3 modalities',  X_multi_64,                                   y_64, True),
]

all_results = []

for exp_name, X, y_df, use_loo in EXPERIMENTS:
    print(f'\n{"="*58}')
    print(f'{exp_name}  (n={len(X)}, LOO-CV)')
    print(f'{"="*58}')

    for label in LABEL_COLS:
        if label not in y_df.columns: continue
        y_lbl = y_df[label].reindex(X.index).dropna()
        pos   = int(y_lbl.sum())
        print(f'\n  {label}  (pos={pos}/{len(y_lbl)})')
        if pos < 3:
            print('    Skipping — too few positives')
            continue

        # Standard models
        for mname, model in make_models().items():
            m = run_cv(X, y_df[label], model, use_loo=True)
            if m:
                print(f'    {mname:<12} AUC={m["AUC"]:.3f}  F1={m["F1"]:.3f}'
                      f'  BalAcc={m["BalAcc"]:.3f}  MCC={m["MCC"]:.3f}')
                all_results.append({'experiment': exp_name, 'label': label,
                                    'model': mname, **m})

        # LateFusion for experiment 7 only
        if '7.' in exp_name:
            n_c   = X_clin_64.shape[1]
            n_g   = X_genomic_64.shape[1]
            y_arr = y_df[label].reindex(X.index).dropna().astype(int).values
            X_arr = X.loc[y_df[label].reindex(X.index).dropna().index].values
            y_prob = np.zeros(len(y_arr))
            for tr_idx, te_idx in LeaveOneOut().split(X_arr):
                if len(np.unique(y_arr[tr_idx])) < 2:
                    y_prob[te_idx] = 0.5
                    continue
                try:
                    lf = LateFusion(n_clin=n_c, n_gen=n_g, random_state=42)
                    lf.fit(X_arr[tr_idx], y_arr[tr_idx])
                    y_prob[te_idx] = lf.predict_proba(X_arr[te_idx].reshape(1,-1))[:, 1]
                except:
                    y_prob[te_idx] = 0.5
            m = evaluate(y_arr, y_prob)
            print(f'    {"LateFusion":<12} AUC={m["AUC"]:.3f}  F1={m["F1"]:.3f}'
                  f'  BalAcc={m["BalAcc"]:.3f}  MCC={m["MCC"]:.3f}')
            all_results.append({'experiment': exp_name, 'label': label,
                                'model': 'LateFusion', **m})

results_df = pd.DataFrame(all_results)
results_df.to_csv(f'{OUT}/results_final_64.csv', index=False)
print('\n\nAll experiments done! Saved to results_final_64.csv')

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import timm
warnings.filterwarnings('ignore')

CLEAN_TILES = '/kaggle/input/datasets/mdafsar07/clean-tiles/tiles1_clean'
OUT         = '/kaggle/working'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Try pathology-specific models, fallback to best available ─────────────────
def load_best_model():
    """Try models in order of quality for histopathology."""
    
    # Option 1: EfficientNet-B4 (better than ResNet50 for medical imaging)
    try:
        model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
        model = model.to(device).eval()
        print('✅ Using EfficientNet-B4 (much better than ResNet50 for pathology)')
        transform = transforms.Compose([
            transforms.Resize((380, 380)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                                 std =[0.229,0.224,0.225])
        ])
        feat_dim = 1792
        return model, transform, feat_dim, 'EfficientNet-B4'
    except Exception as e:
        print(f'EfficientNet failed: {e}')
    
    # Option 2: DenseNet121 (used in many pathology papers)
    try:
        model = timm.create_model('densenet121', pretrained=True, num_classes=0)
        model = model.to(device).eval()
        print('✅ Using DenseNet121')
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                                 std =[0.229,0.224,0.225])
        ])
        feat_dim = 1024
        return model, transform, feat_dim, 'DenseNet121'
    except Exception as e:
        print(f'DenseNet failed: {e}')
    
    # Option 3: ResNet50 (fallback)
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model = torch.nn.Sequential(*list(model.children())[:-1])
    model = model.to(device).eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],
                             std =[0.229,0.224,0.225])
    ])
    print('✅ Using ResNet50 (fallback)')
    return model, transform, 2048, 'ResNet50'

feat_model, transform, feat_dim, model_name = load_best_model()
print(f'Feature dim: {feat_dim}')

In [ ]:
import os
from collections import Counter

print('CLEAN_TILES =', CLEAN_TILES)

slides = [s for s in os.listdir(CLEAN_TILES) if os.path.isdir(f'{CLEAN_TILES}/{s}')]
print('Slide folders:', len(slides))
print('Unique patients (from folder names):', len({'-'.join(s.split('-')[:3]) for s in slides}))

# count every image file at any depth, by extension
ext = Counter()
for root, _, files in os.walk(CLEAN_TILES):
    for f in files:
        ext[os.path.splitext(f)[1].lower()] += 1
print('File types:', dict(ext))

# show what the first slide folder looks like
s0 = f'{CLEAN_TILES}/{slides[0]}'
print('First slide contents:', os.listdir(s0)[:5])

In [ ]:
import os
from collections import Counter

CLEAN_TILES = '/kaggle/input/datasets/mdafsar07/final-histo/Final_Clean_Tiles'
print('CLEAN_TILES =', CLEAN_TILES)

slides = [s for s in os.listdir(CLEAN_TILES) if os.path.isdir(f'{CLEAN_TILES}/{s}')]
pids = {'-'.join(s.split('-')[:3]) for s in slides}
print('Slide folders:', len(slides))
print('Unique patients:', len(pids))
print('Sample types (01=tumor, 11=normal):', Counter(s.split('-')[3][:2] for s in slides))

ext = Counter()
for root, _, files in os.walk(CLEAN_TILES):
    for f in files:
        ext[os.path.splitext(f)[1].lower()] += 1
print('File types:', dict(ext))
print('First slide contents:', os.listdir(f'{CLEAN_TILES}/{slides[0]}')[:2])

In [ ]:
tumor_slides = [s for s in slides if s.split('-')[3].startswith('01')]
tumor_pids   = {'-'.join(s.split('-')[:3]) for s in tumor_slides}
cohort_ids   = set(X_clin.index) & set(X_genomic.index)

print('Tumor slides            :', len(tumor_slides))
print('Patients with tumor     :', len(tumor_pids))
print('Normal-only patients    :', len(pids - tumor_pids), '(dropped)')
print('Overlap with cohort     :', len(tumor_pids & cohort_ids))

In [ ]:
tumor_ids = sorted(tumor_pids & set(X_clin.index) & set(X_genomic.index))
yy = y_full.loc[tumor_ids]
for c in yy.columns:
    vc = yy[c].value_counts(dropna=False).to_dict()
    print(f'{c:28s}', vc)

In [ ]:
import os
from collections import Counter

print('CLEAN_TILES =', CLEAN_TILES)

slides = [s for s in os.listdir(CLEAN_TILES) if os.path.isdir(f'{CLEAN_TILES}/{s}')]
print('Slide folders:', len(slides))
print('Unique patients (from folder names):', len({'-'.join(s.split('-')[:3]) for s in slides}))

# count every image file at any depth, by extension
ext = Counter()
for root, _, files in os.walk(CLEAN_TILES):
    for f in files:
        ext[os.path.splitext(f)[1].lower()] += 1
print('File types:', dict(ext))

# show what the first slide folder looks like
s0 = f'{CLEAN_TILES}/{slides[0]}'
print('First slide contents:', os.listdir(s0)[:5])

In [ ]:
class AttentionMIL(nn.Module):
    """
    Attention-based Multiple Instance Learning for WSI tiles.
    Each patient = bag of tiles. Model learns WHICH tiles matter most.
    This is the standard architecture for pathology in 2024-2026 papers.
    Much better than simple averaging of tile embeddings.
    """
    def __init__(self, feat_dim=512, hidden_dim=128, n_classes=1):
        super().__init__()
        
        # Feature compressor
        self.compressor = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, hidden_dim),
            nn.ReLU(),
        )
        
        # Attention network — learns which tiles to focus on
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, n_classes)
        )
    
    def forward(self, x):
        # x: (n_tiles, feat_dim)
        h = self.compressor(x)           # (n_tiles, hidden_dim)
        a = self.attention(h)            # (n_tiles, 1)
        a = torch.softmax(a, dim=0)      # normalize weights
        z = (a * h).sum(dim=0)           # weighted aggregation
        out = self.classifier(z)         # prediction
        return out, a                    # return attention weights too!

# Test it
test_input = torch.randn(50, feat_dim).to(device)  # 50 tiles
mil_model = AttentionMIL(feat_dim=feat_dim).to(device)
out, attn = mil_model(test_input)
print(f'MIL model output: {out.shape}')
print(f'Attention weights: {attn.shape}  (one per tile)')
print(f'Top 3 tile weights: {attn.squeeze().topk(3).values.detach().cpu().numpy()}')
print('AttentionMIL model ready!')

In [ ]:
# ── Fast AttentionMIL — precompute all embeddings first ───────────────────────
# Instead of reloading tiles 64 times, load all tiles ONCE then do LOO-CV
# This reduces 64 × tile_loading to 1 × tile_loading = 64x faster

import torch
import torch.nn as nn
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score
import numpy as np
from tqdm import tqdm

print('Step 1: Precomputing all tile embeddings (one time only)...')

MAX_TILES_PER_PATIENT = 100

# Build tile paths per patient
tile_dict = {}
for pid in manifest['patient_id'].unique():
    paths = manifest[manifest['patient_id']==pid]['tile_path'].tolist()
    paths = paths[:MAX_TILES_PER_PATIENT]
    tile_dict[pid] = paths

# Precompute embeddings for ALL patients at once
all_embeddings = {}  # pid → tensor (n_tiles, feat_dim)

feat_model.eval()
for pid in tqdm(tile_dict.keys(), desc='Precomputing embeddings'):
    paths = tile_dict[pid]
    tensors = []
    for p in paths:
        try:
            tensors.append(transform(Image.open(p).convert('RGB')))
        except: continue
    if not tensors:
        continue
    batch = torch.stack(tensors).to(device)
    with torch.no_grad():
        embs = feat_model(batch)
        if len(embs.shape) > 2:
            embs = embs.squeeze(-1).squeeze(-1)
    all_embeddings[pid] = embs.cpu()  # store on CPU to save GPU memory

print(f'Embeddings precomputed for {len(all_embeddings)} patients')
print(f'Embedding dim: {list(all_embeddings.values())[0].shape}')

In [ ]:
# ── AttentionMIL (simplified, fast version) ───────────────────────────────────
class FastAttentionMIL(nn.Module):
    def __init__(self, feat_dim=512, hidden_dim=64):
        super().__init__()
        self.compress = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = self.compress(x)
        a = torch.softmax(self.attention(h), dim=0)
        z = (a * h).sum(dim=0, keepdim=True)
        return self.classifier(z).squeeze(), a

print('FastAttentionMIL defined')

In [ ]:
import pandas as pd
import numpy as np

BRCA_BASE = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'
OUT       = '/kaggle/working'

# Load raw clinical for stage/node columns
X_clin_raw = pd.read_csv(f'{BRCA_BASE}/X_clinical.csv', index_col='patient_id')
y_old      = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')

stage   = X_clin_raw['ord_ajcc_pathologic_stage']
n_stage = X_clin_raw['ord_ajcc_pathologic_n']

# Cancer stage label (Early=0 vs Late=1)
label_stage = pd.Series(np.nan, index=X_clin_raw.index)
label_stage[stage >= 8] = 1
label_stage[(stage >= 1) & (stage <= 6)] = 0

# Node positive label
label_node = pd.Series(np.nan, index=X_clin_raw.index)
label_node[n_stage >= 6] = 1
label_node[(n_stage >= 2) & (n_stage <= 5)] = 0

y_4tasks = pd.DataFrame({
    'label_cancer_stage'      : label_stage,
    'label_node_positive'     : label_node,
    'label_survival'          : y_old['label_survival'],
    'label_recurrence'        : y_old['label_recurrence'],
    'label_treatment_response': y_old['label_treatment_response'],
}, index=X_clin_raw.index)

y_4tasks.to_csv(f'{OUT}/y_labels_4tasks.csv')
print(f'y_4tasks rebuilt: {y_4tasks.shape}')
print(y_4tasks.notna().sum())

In [ ]:
# ── LOO-CV with precomputed embeddings ────────────────────────────────────────
def run_mil_loo(all_embeddings, y_series, n_epochs=15, lr=1e-3):
    y = y_series.reindex(list(all_embeddings.keys())).dropna().astype(float)
    valid_pids = list(y.index)

    if y.sum() < 3:
        print('Too few positives')
        return None, None, None

    pos_weight = torch.tensor([(y==0).sum() / max(y.sum(),1)]).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    y_probs    = np.zeros(len(valid_pids))
    attn_store = {}

    for i, test_pid in enumerate(tqdm(valid_pids, desc='LOO-CV')):
        train_pids = [p for p in valid_pids if p != test_pid]

        # Fresh model each fold
        model = FastAttentionMIL(
            feat_dim=list(all_embeddings.values())[0].shape[1],
            hidden_dim=64
        ).to(device)
        optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-3)

        # Train
        model.train()
        for epoch in range(n_epochs):
            for pid in train_pids:
                embs  = all_embeddings[pid].to(device)
                label = torch.tensor([y[pid]], dtype=torch.float).to(device)
                optimizer.zero_grad()
                out, _ = model(embs)
                loss   = criterion(out.unsqueeze(0), label)
                loss.backward()
                optimizer.step()

        # Predict
        model.eval()
        with torch.no_grad():
            embs = all_embeddings[test_pid].to(device)
            out, attn = model(embs)
            prob = torch.sigmoid(out).item()
            y_probs[i] = prob
            attn_store[test_pid] = {
                'weights': attn.squeeze().cpu().numpy(),
                'paths'  : tile_dict[test_pid]
            }

    return y_probs, y, attn_store

# ── Run for all 5 labels ──────────────────────────────────────────────────────
BRCA_BASE = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'
OUT       = '/kaggle/working'
y_old     = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')
y_4tasks  = pd.read_csv(f'{OUT}/y_labels_4tasks.csv',   index_col='patient_id')

MIL_RESULTS = {}
LABEL_COLS  = ['label_cancer_stage','label_node_positive',
               'label_survival','label_recurrence','label_treatment_response']
attn_stores = {}

for label in LABEL_COLS:
    y_src = y_4tasks if label in y_4tasks.columns else y_old
    print(f'\n{"="*50}')
    print(f'AttentionMIL → {label}')
    print(f'{"="*50}')

    y_probs, y_true, attn = run_mil_loo(
        all_embeddings=all_embeddings,
        y_series=y_src[label],
        n_epochs=15,
        lr=1e-3
    )

    if y_probs is not None:
        auc  = roc_auc_score(y_true.values, y_probs)
        pred = (y_probs >= 0.5).astype(int)
        f1   = f1_score(y_true.values, pred, zero_division=0)
        bal  = balanced_accuracy_score(y_true.values, pred)
        mcc  = matthews_corrcoef(y_true.values, pred)
        print(f'  AUC={auc:.3f}  F1={f1:.3f}  BalAcc={bal:.3f}  MCC={mcc:.3f}')
        MIL_RESULTS[label] = {'AUC':round(auc,3),'F1':round(f1,3),
                               'BalAcc':round(bal,3),'MCC':round(mcc,3)}
        attn_stores[label] = attn

print('\n\nAll MIL done!')
print('\nMIL Summary:')
for label, m in MIL_RESULTS.items():
    print(f'  {label:<38} AUC={m["AUC"]:.3f}  F1={m["F1"]:.3f}')

In [ ]:
# ── Compare ALL approaches cleanly ────────────────────────────────────────────
import pandas as pd

results_summary = {
    'Approach': [
        'Clinical only (ElasticNet)',
        'Genomic only (ElasticNet)',
        'Histopath only (ElasticNet)',
        'Clinical+Genomic (Ensemble)',
        'Clinical+Histo (LightGBM)',
        'Genomic+Histo (XGBoost)',
        'All 3 - Concat (ElasticNet)',
        'All 3 - LateFusion',
        'All 3 - AttentionMIL',
    ],
    'n_patients': [1095,1095,64,1095,64,64,64,64,64],
    'CV': ['5-fold','5-fold','LOO','5-fold','LOO','LOO','LOO','LOO','LOO'],
    'Survival_AUC':   [0.730,0.578,0.652,0.730,0.530,0.844,0.889,0.896,0.342],
    'Recurrence_AUC': [0.739,0.568,0.467,0.784,0.806,0.522,0.652,0.822,0.134],
    'Treatment_AUC':  [0.701,0.624,0.474,0.700,0.496,0.415,0.393,0.401,0.575],
}

df_summary = pd.DataFrame(results_summary)
df_summary.to_csv(f'{OUT}/final_comparison.csv', index=False)

print('='*75)
print('COMPLETE METHOD COMPARISON')
print('='*75)
print(df_summary.to_string(index=False))

print('\n\nKEY FINDING:')
print('  AttentionMIL underperforms because:')
print('  1. Only 64 patients (too few for deep learning)')
print('  2. ImageNet features not suited for histopathology')
print('  3. Need UNI/CONCH pathology foundation models')
print('\n  Classical ML (LateFusion) is better for this dataset size')
print('  LateFusion Survival AUC=0.896 > AttentionMIL AUC=0.342')

In [ ]:
!pip install efficientnet_pytorch -q
print('Done')

In [ ]:
# ── GradCAM using already-loaded feat_model ───────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

class SimpleGradCAM:
    def __init__(self, model):
        self.model       = model
        self.gradients   = None
        self.activations = None
        # Hook onto last block of EfficientNet
        target = list(model.children())[-3]  # _blocks last layer
        target.register_forward_hook(
            lambda m,i,o: setattr(self, 'activations', o.detach()))
        target.register_full_backward_hook(
            lambda m,i,o: setattr(self, 'gradients', o[0].detach()))

    def __call__(self, img_tensor):
        self.model.eval()
        out = self.model(img_tensor)
        self.model.zero_grad()
        out.sum().backward()
        w   = self.gradients.mean(dim=[2,3], keepdim=True)
        cam = (w * self.activations).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = SimpleGradCAM(feat_model)

def show_gradcam(tile_paths, patient_id, n_tiles=4):
    fig, axes = plt.subplots(2, n_tiles, figsize=(4*n_tiles, 8))
    fig.suptitle(f'GradCAM — Patient: {patient_id}', 
                 fontsize=14, fontweight='bold')

    for col, path in enumerate(tile_paths[:n_tiles]):
        try:
            img = np.array(Image.open(path).convert('RGB'))
            axes[0,col].imshow(img)
            axes[0,col].set_title('Original Tile')
            axes[0,col].axis('off')

            t = transform(Image.open(path).convert('RGB'))
            t = t.unsqueeze(0).to(device)
            t.requires_grad_(True)
            cam = gradcam(t)

            cam_resized = np.array(
                Image.fromarray((cam*255).astype(np.uint8)).resize(
                    (img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0
            overlay = np.clip(
                0.5 * cm.jet(cam_resized)[:,:,:3] + 0.5 * img/255.0, 0, 1)

            axes[1,col].imshow(overlay)
            axes[1,col].set_title('GradCAM Overlay')
            axes[1,col].axis('off')
        except Exception as e:
            print(f'  Tile error: {e}')

    plt.tight_layout()
    save_path = f'{OUT}/gradcam_{patient_id}.png'
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

# Run GradCAM on first 3 patients
OUT = '/kaggle/working'
sample_pids = list(tile_dict.keys())[:3]

for pid in sample_pids:
    paths = tile_dict[pid][:4]
    print(f'Generating GradCAM for {pid}...')
    show_gradcam(paths, pid)

print('GradCAM done!')

In [ ]:
# ── Improved GradCAM with tissue-focused tiles only ───────────────────────────
import cv2

def is_tissue_tile(path, white_thresh=0.4):
    """Filter out background/white tiles."""
    img = np.array(Image.open(path).convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    white_pct = np.sum(gray > 220) / gray.size
    return white_pct < white_thresh

def show_gradcam_improved(tile_paths, patient_id, label_name='', n_tiles=4):
    # Filter tissue tiles only
    tissue_tiles = [p for p in tile_paths if is_tissue_tile(p)]
    if not tissue_tiles:
        tissue_tiles = tile_paths
    print(f'  {len(tissue_tiles)}/{len(tile_paths)} tissue tiles')

    fig, axes = plt.subplots(2, n_tiles, figsize=(4*n_tiles, 8))
    fig.suptitle(
        f'GradCAM — Patient: {patient_id}  ({label_name})',
        fontsize=14, fontweight='bold'
    )

    for col, path in enumerate(tissue_tiles[:n_tiles]):
        try:
            img = np.array(Image.open(path).convert('RGB'))

            # GradCAM
            t = transform(Image.open(path).convert('RGB'))
            t = t.unsqueeze(0).to(device)
            t.requires_grad_(True)
            cam = gradcam(t)

            # Better colormap — use hot instead of jet
            cam_resized = np.array(
                Image.fromarray((cam*255).astype(np.uint8)).resize(
                    (img.shape[1], img.shape[0]), Image.BILINEAR)
            ) / 255.0

            # Show original
            axes[0,col].imshow(img)
            axes[0,col].set_title(f'H&E Tile {col+1}', fontsize=10)
            axes[0,col].axis('off')

            # Overlay with 'hot' colormap — more visible on H&E
            colormap = cm.hot(cam_resized)[:,:,:3]
            overlay  = np.clip(0.45*colormap + 0.55*img/255.0, 0, 1)
            axes[1,col].imshow(overlay)
            axes[1,col].set_title(f'GradCAM (max={cam.max():.2f})', fontsize=10)
            axes[1,col].axis('off')

        except Exception as e:
            print(f'  Error: {e}')
            for row in range(2):
                axes[row,col].axis('off')

    plt.tight_layout()
    fname = f'{OUT}/gradcam_{patient_id}_{label_name}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

# ── Show for positive vs negative patients ────────────────────────────────────
y_survival = pd.read_csv(
    f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')['label_survival']

# Pick 2 positive (died) and 2 negative (alive) patients
pids_all = list(tile_dict.keys())
pos_pids = [p for p in pids_all if y_survival.get(p, np.nan) == 1][:2]
neg_pids = [p for p in pids_all if y_survival.get(p, np.nan) == 0][:2]

print('=== POSITIVE patients (Survival=1, Died) ===')
for pid in pos_pids:
    show_gradcam_improved(tile_dict[pid], pid, label_name='Died')

print('\n=== NEGATIVE patients (Survival=0, Alive) ===')
for pid in neg_pids:
    show_gradcam_improved(tile_dict[pid], pid, label_name='Alive')

In [ ]:
!pip install timm -q
print('Done')

In [ ]:
import timm
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import cv2

OUT       = '/kaggle/working'
BRCA_BASE = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'

# ── Load ViT pretrained on pathology-like images ──────────────────────────────
print('Loading ViT-Small...')
vit_model = timm.create_model(
    'vit_small_patch16_224',
    pretrained=True,
    num_classes=0
)
vit_model = vit_model.to(device).eval()
print(f'ViT feature dim: {vit_model.num_features}')

vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

def is_tissue(path, thresh=0.4):
    img  = np.array(Image.open(path).convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return np.sum(gray > 220)/gray.size < thresh

# ── Extract ViT embeddings ────────────────────────────────────────────────────
vit_embs = {}
for pid in tqdm(tile_dict.keys(), desc='ViT'):
    paths   = tile_dict[pid]
    good    = [p for p in paths if is_tissue(p)]
    if not good: good = paths[:50]
    tensors = []
    for p in good[:100]:
        try: tensors.append(vit_transform(Image.open(p).convert('RGB')))
        except: continue
    if not tensors: continue
    batch = torch.stack(tensors).to(device)
    with torch.no_grad():
        embs = vit_model(batch)
    vit_embs[pid] = embs.mean(dim=0).cpu().numpy()

print(f'Done: {len(vit_embs)} patients, dim={list(vit_embs.values())[0].shape}')

# PCA
vit_raw = pd.DataFrame(vit_embs).T
vit_raw.index.name = 'patient_id'
N = min(50, len(vit_raw)-1)
sc  = StandardScaler()
pc  = PCA(n_components=N, random_state=42)
histo_vit = pd.DataFrame(
    pc.fit_transform(sc.fit_transform(vit_raw)),
    index=vit_raw.index,
    columns=[f'vit_PC{i+1}' for i in range(N)]
)
print(f'ViT PCA: {histo_vit.shape}  Var: {pc.explained_variance_ratio_.sum():.1%}')
histo_vit.to_csv(f'{OUT}/X_histopath_vit.csv')

In [ ]:
import os
# Check what histopath files exist
files = [f for f in os.listdir('/kaggle/working') if 'histo' in f.lower() or 'pca' in f.lower()]
print('Histopath files in /kaggle/working:')
for f in files:
    size = os.path.getsize(f'/kaggle/working/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')

In [ ]:
import os, glob

print('=== Current notebook ===')
print(os.getcwd())

print('\n=== Datasets available ===')
for d in os.listdir('/kaggle/input/datasets'):
    print(f'  {d}')
    for item in os.listdir(f'/kaggle/input/datasets/{d}'):
        print(f'    {item}')

print('\n=== Files in /kaggle/working ===')
for f in os.listdir('/kaggle/working'):
    if not f.startswith('.'):
        size = os.path.getsize(f'/kaggle/working/{f}') / 1024
        print(f'  {f}  ({size:.1f} KB)')

print('\n=== Clean_Tiles path ===')
tiles = glob.glob('/kaggle/input/**/*.png', recursive=True)
print(f'Total PNG files: {len(tiles)}')
if tiles:
    print(f'Sample: {tiles[0]}')

In [ ]:
import os
import torch

print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Clean_Tiles exists: {os.path.exists("/kaggle/input/datasets/mdafsar07/clean-tiles/tiles1_clean")}')
print(f'BRCA_BASE exists  : {os.path.exists("/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset")}')
print(f'Working dir empty : {os.listdir("/kaggle/working")}')
print('\nReady to run!')

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import timm
import cv2
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut, cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, matthews_corrcoef
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, ClassifierMixin
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
CLEAN_TILES = '/kaggle/input/datasets/mdafsar07/clean-tiles/tiles1_clean'
BRCA_BASE   = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'
OUT         = '/kaggle/working'
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

LABEL_COLS  = ['label_cancer_stage','label_node_positive',
               'label_survival','label_recurrence','label_treatment_response']

LEAK_COLS = [
    'fu.follow_ups.days_to_recurrence','extra_days_to_recurrence',
    'extra_days_to_progression','demographic.days_to_death',
    'diagnoses.days_to_last_follow_up','fu.follow_ups.days_to_follow_up',
    'disease_response_wt-with tumor','disease_response_tf-tumor free',
    'disease_response_unknown','disease_response_wt-with tumor.1',
    'disease_response_tf-tumor free.1','disease_response_unknown.1',
    'fu.follow_ups.disease_response','extra_disease_response_tf-tumor free',
    'extra_disease_response_wt-with tumor','extra_disease_response_unknown',
    'progression_or_recurrence_yes','progression_or_recurrence_unknown',
    'ord_ajcc_pathologic_stage','ord_ajcc_pathologic_n',
    'ord_ajcc_pathologic_m','ord_ajcc_pathologic_t','ord_tumor_grade',
    'progression_or_recurrence_type_distant',
    'progression_or_recurrence_type_locoregional',
    'progression_or_recurrence_type_unknown',
]

def clean_X(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object or df[col].dtype == bool:
            df[col] = df[col].map(
                {'True':1,'False':0,True:1,False:0,
                 'true':1,'false':0,'yes':1,'no':0}
            ).fillna(df[col])
            df[col] = pd.to_numeric(df[col], errors='coerce')
    all_nan = df.columns[df.isna().all()].tolist()
    if all_nan: df = df.drop(columns=all_nan)
    imp = SimpleImputer(strategy='median')
    return pd.DataFrame(imp.fit_transform(df),
                        index=df.index, columns=df.columns)

def remove_leaks(df):
    return df.drop(columns=[c for c in LEAK_COLS if c in df.columns])

print('Loading clinical+genomic...')
X_clin_raw = pd.read_csv(f'{BRCA_BASE}/X_clinical.csv', index_col='patient_id')
X_clin     = clean_X(remove_leaks(X_clin_raw))
X_pca      = clean_X(remove_leaks(
    pd.read_csv(f'{BRCA_BASE}/X_pca_v2.csv', index_col='patient_id')))
X_genomic  = clean_X(X_pca.filter(like='PC'))
y_old      = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')

# Build 5-task labels
stage   = X_clin_raw['ord_ajcc_pathologic_stage']
n_stage = X_clin_raw['ord_ajcc_pathologic_n']
label_stage = pd.Series(np.nan, index=X_clin_raw.index)
label_stage[stage >= 8] = 1
label_stage[(stage >= 1) & (stage <= 6)] = 0
label_node = pd.Series(np.nan, index=X_clin_raw.index)
label_node[n_stage >= 6] = 1
label_node[(n_stage >= 2) & (n_stage <= 5)] = 0
y_full = pd.DataFrame({
    'label_cancer_stage'      : label_stage,
    'label_node_positive'     : label_node,
    'label_survival'          : y_old['label_survival'],
    'label_recurrence'        : y_old['label_recurrence'],
    'label_treatment_response': y_old['label_treatment_response'],
}, index=X_clin_raw.index)
y_full.to_csv(f'{OUT}/y_labels_4tasks.csv')
print(f'Clinical: {X_clin.shape}  Genomic: {X_genomic.shape}  Labels: {y_full.shape}')

In [ ]:
#  Build tile manifest 
print('\nBuilding tile manifest...')
records = []
for slide in os.listdir(CLEAN_TILES):
    slide_dir = f'{CLEAN_TILES}/{slide}'
    if not os.path.isdir(slide_dir): continue
    for tile in os.listdir(slide_dir):
        if tile.endswith('.png'):
            records.append({
                'patient_id': '-'.join(slide.split('-')[:3]),
                'tile_path' : f'{slide_dir}/{tile}'
            })

manifest = pd.DataFrame(records)
manifest = (manifest.groupby('patient_id')
                    .apply(lambda x: x.sample(min(len(x),150), random_state=42))
                    .reset_index(drop=True))
tile_dict = {pid: grp['tile_path'].tolist()
             for pid, grp in manifest.groupby('patient_id')}
print(f'Patients: {len(tile_dict)}  Tiles: {len(manifest):,}')

In [2]:
!pip install efficientnet_pytorch timm -q
print('Done')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Done


In [18]:
#  Extract BOTH EfficientNet AND ViT embeddings 
def is_tissue(path, thresh=0.4):
    try:
        img  = np.array(Image.open(path).convert('RGB'))
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return np.sum(gray > 220)/gray.size < thresh
    except: return True

def extract_embeddings(model, transform, tile_dict, model_name, max_tiles=100):
    model.eval()
    emb_dict = {}
    for pid in tqdm(tile_dict.keys(), desc=f'{model_name}'):
        paths   = tile_dict[pid]
        good    = [p for p in paths if is_tissue(p)]
        if not good: good = paths
        tensors = []
        for p in good[:max_tiles]:
            try: tensors.append(transform(Image.open(p).convert('RGB')))
            except: continue
        if not tensors: continue
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            embs = model(batch)
            if len(embs.shape) > 2:
                embs = embs.squeeze(-1).squeeze(-1)
        emb_dict[pid] = embs.mean(dim=0).cpu().numpy()
    return emb_dict

def make_pca(emb_dict, n_pcs=50, prefix='PC'):
    raw = pd.DataFrame(emb_dict).T
    raw.index.name = 'patient_id'
    N   = min(n_pcs, len(raw)-1)
    sc  = StandardScaler()
    pc  = PCA(n_components=N, random_state=42)
    pca_df = pd.DataFrame(
        pc.fit_transform(sc.fit_transform(raw)),
        index=raw.index,
        columns=[f'{prefix}{i+1}' for i in range(N)]
    )
    var = pc.explained_variance_ratio_.sum()
    return pca_df, var

# EfficientNet-B4
print('\n--- EfficientNet-B4 ---')
from efficientnet_pytorch import EfficientNet as EfficientNetModel
eff_net = EfficientNetModel.from_pretrained('efficientnet-b4')
eff_net._fc = torch.nn.Identity()
eff_net = eff_net.to(device).eval()
eff_transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
eff_embs = extract_embeddings(eff_net, eff_transform, tile_dict, 'EfficientNet-B4')
eff_pca, eff_var = make_pca(eff_embs, n_pcs=50, prefix='eff_PC')
eff_pca.to_csv(f'{OUT}/X_histo_eff.csv')
print(f'EfficientNet PCA: {eff_pca.shape}  Var: {eff_var:.1%}')
del eff_net  # free GPU memory

# ViT-Small
print('\n--- ViT-Small ---')
vit = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
vit = vit.to(device).eval()
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])
vit_embs = extract_embeddings(vit, vit_transform, tile_dict, 'ViT-Small')
vit_pca, vit_var = make_pca(vit_embs, n_pcs=50, prefix='vit_PC')
vit_pca.to_csv(f'{OUT}/X_histo_vit.csv')
print(f'ViT PCA: {vit_pca.shape}  Var: {vit_var:.1%}')
del vit

print('\nBoth models extracted and saved!')


--- EfficientNet-B4 ---
Loaded pretrained weights for efficientnet-b4


NameError: name 'torch' is not defined

In [ ]:
# Quick LOO-AUC comparison 
LOO = LeaveOneOut()

def quick_auc(X, y_series):
    y = y_series.reindex(X.index).dropna().astype(int)
    X_v = X.loc[y.index]
    if y.sum() < 3: return 'N/A'
    k = min(3, int(y.sum())-1)
    pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=42, k_neighbors=k)),
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(penalty='elasticnet', solver='saga',
                       l1_ratio=0.5, C=0.1, class_weight='balanced',
                       max_iter=2000, random_state=42))
    ])
    try:
        prob = cross_val_predict(pipe, X_v, y, cv=LOO,
                                  method='predict_proba')[:,1]
        return round(roc_auc_score(y, prob), 3)
    except: return 'ERR'

y_4tasks = pd.read_csv(f'{OUT}/y_labels_4tasks.csv', index_col='patient_id')

print('\n' + '='*65)
print('FEATURE COMPARISON (LOO-CV, ElasticNet, 64 patients)')
print('='*65)
print(f'{"Label":<38} {"EfficientNet":>13} {"ViT-Small":>10} {"Best":>5}')
print('-'*65)

comparison = []
for label in LABEL_COLS:
    auc_eff = quick_auc(eff_pca, y_4tasks[label])
    auc_vit = quick_auc(vit_pca, y_4tasks[label])
    if isinstance(auc_eff, float) and isinstance(auc_vit, float):
        best = 'ViT' if auc_vit > auc_eff else 'Eff'
    else:
        best = '-'
    print(f'{label:<38} {str(auc_eff):>13} {str(auc_vit):>10} {best:>5}')
    comparison.append({'label':label,'EfficientNet':auc_eff,'ViT':auc_vit,'best':best})

pd.DataFrame(comparison).to_csv(f'{OUT}/feature_comparison.csv', index=False)
print('\nSaved feature_comparison.csv')

In [ ]:
#Build ensemble histo features (EfficientNet + ViT combined)
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, matthews_corrcoef
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, ClassifierMixin
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

OUT       = '/kaggle/working'
BRCA_BASE = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'
LABEL_COLS = ['label_cancer_stage','label_node_positive',
              'label_survival','label_recurrence','label_treatment_response']

# Load saved PCAs
eff_pca = pd.read_csv(f'{OUT}/X_histo_eff.csv', index_col='patient_id')
vit_pca = pd.read_csv(f'{OUT}/X_histo_vit.csv', index_col='patient_id')
y_4tasks = pd.read_csv(f'{OUT}/y_labels_4tasks.csv', index_col='patient_id')

# Combine EfficientNet + ViT features
histo_ensemble = eff_pca.join(vit_pca, how='inner', lsuffix='_eff', rsuffix='_vit')
print(f'Ensemble histo features: {histo_ensemble.shape}')
# Reduce combined features with PCA
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
sc  = StandardScaler()
pc  = PCA(n_components=50, random_state=42)
histo_ens_arr = pc.fit_transform(sc.fit_transform(
    imp.fit_transform(histo_ensemble)))
histo_combined = pd.DataFrame(
    histo_ens_arr,
    index=histo_ensemble.index,
    columns=[f'ens_PC{i+1}' for i in range(50)]
)
print(f'Ensemble PCA: {histo_combined.shape}  Var: {pc.explained_variance_ratio_.sum():.1%}')
histo_combined.to_csv(f'{OUT}/X_histo_ensemble.csv')

# Load clinical+genomic subsets for 64 patients
LEAK_COLS = [
    'fu.follow_ups.days_to_recurrence','extra_days_to_recurrence',
    'extra_days_to_progression','demographic.days_to_death',
    'diagnoses.days_to_last_follow_up','fu.follow_ups.days_to_follow_up',
    'disease_response_wt-with tumor','disease_response_tf-tumor free',
    'disease_response_unknown','disease_response_wt-with tumor.1',
    'disease_response_tf-tumor free.1','disease_response_unknown.1',
    'fu.follow_ups.disease_response','extra_disease_response_tf-tumor free',
    'extra_disease_response_wt-with tumor','extra_disease_response_unknown',
    'progression_or_recurrence_yes','progression_or_recurrence_unknown',
    'ord_ajcc_pathologic_stage','ord_ajcc_pathologic_n',
    'ord_ajcc_pathologic_m','ord_ajcc_pathologic_t','ord_tumor_grade',
]

def clean_X(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object or df[col].dtype == bool:
            df[col] = df[col].map(
                {'True':1,'False':0,True:1,False:0}
            ).fillna(df[col])
            df[col] = pd.to_numeric(df[col], errors='coerce')
    all_nan = df.columns[df.isna().all()].tolist()
    if all_nan: df = df.drop(columns=all_nan)
    imp = SimpleImputer(strategy='median')
    return pd.DataFrame(imp.fit_transform(df),
                        index=df.index, columns=df.columns)

X_clin  = clean_X(pd.read_csv(f'{BRCA_BASE}/X_clinical.csv',
                               index_col='patient_id').drop(
    columns=[c for c in LEAK_COLS
             if c in pd.read_csv(f'{BRCA_BASE}/X_clinical.csv',
                                  index_col='patient_id').columns]))
X_pca   = clean_X(pd.read_csv(f'{BRCA_BASE}/X_pca_v2.csv',
                               index_col='patient_id').drop(
    columns=[c for c in LEAK_COLS
             if c in pd.read_csv(f'{BRCA_BASE}/X_pca_v2.csv',
                                  index_col='patient_id').columns]))
X_genomic = clean_X(X_pca.filter(like='PC'))
y_full    = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')

idx_64       = list(histo_combined.index)
X_clin_64    = X_clin.loc[idx_64]
X_genomic_64 = X_genomic.loc[idx_64]
X_pca_64     = X_pca.loc[idx_64]
y_64         = y_4tasks.loc[idx_64]

# Best histo per label (pick winner from comparison)
BEST_HISTO = {
    'label_cancer_stage'     : vit_pca,    # ViT won
    'label_node_positive'    : eff_pca,    # Eff won
    'label_survival'         : eff_pca,    # Eff won
    'label_recurrence'       : vit_pca,    # ViT won
    'label_treatment_response': eff_pca,   # Eff won
}

print('All matrices ready!')
print(f'  X_clin_64    : {X_clin_64.shape}')
print(f'  X_genomic_64 : {X_genomic_64.shape}')
print(f'  X_pca_64     : {X_pca_64.shape}')
print(f'  histo_eff    : {eff_pca.shape}')
print(f'  histo_vit    : {vit_pca.shape}')
print(f'  histo_ens    : {histo_combined.shape}')

In [ ]:
# LateFusion model 
class LateFusion(BaseEstimator, ClassifierMixin):
    def __init__(self, n_clin, n_gen, random_state=42):
        self.n_clin = n_clin
        self.n_gen  = n_gen
        self.random_state = random_state
        self.classes_ = np.array([0,1])

    def _split(self, X):
        return (X[:, :self.n_clin],
                X[:, self.n_clin:self.n_clin+self.n_gen],
                X[:, self.n_clin+self.n_gen:])

    def _expert(self, seed):
        return ImbPipeline([
            ('smote',  SMOTE(random_state=seed, k_neighbors=2)),
            ('scaler', StandardScaler()),
            ('pca',    PCA(n_components=10, random_state=seed)),
            ('clf',    LogisticRegression(penalty='elasticnet', solver='saga',
                           l1_ratio=0.5, C=0.3, class_weight='balanced',
                           max_iter=2000, random_state=seed))
        ])

    def fit(self, X, y):
        X_c, X_g, X_h = self._split(X)
        self.e_c_ = self._expert(self.random_state)
        self.e_g_ = self._expert(self.random_state+1)
        self.e_h_ = self._expert(self.random_state+2)
        self.e_c_.fit(X_c, y)
        self.e_g_.fit(X_g, y)
        self.e_h_.fit(X_h, y)
        p = np.column_stack([
            self.e_c_.predict_proba(X_c)[:,1],
            self.e_g_.predict_proba(X_g)[:,1],
            self.e_h_.predict_proba(X_h)[:,1],
        ])
        self.meta_ = LogisticRegression(C=1.0, class_weight='balanced',
                         max_iter=1000, random_state=self.random_state)
        self.meta_.fit(p, y)
        return self

    def predict_proba(self, X):
        X_c, X_g, X_h = self._split(X)
        p = np.column_stack([
            self.e_c_.predict_proba(X_c)[:,1],
            self.e_g_.predict_proba(X_g)[:,1],
            self.e_h_.predict_proba(X_h)[:,1],
        ])
        return self.meta_.predict_proba(p)

    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

def evaluate(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    try:    auc = roc_auc_score(y_true, y_prob)
    except: auc = 0.5
    return {'AUC'   : round(auc,3),
            'F1'    : round(f1_score(y_true,y_pred,zero_division=0),3),
            'BalAcc': round(balanced_accuracy_score(y_true,y_pred),3),
            'MCC'   : round(matthews_corrcoef(y_true,y_pred),3)}

def run_cv(X, y_series, model, use_loo=False):
    y = y_series.reindex(X.index).dropna().astype(int)
    X_v = X.loc[y.index]
    if y.nunique() < 2 or y.sum() < 3: return None
    k = min(3, int(y.sum())-1)
    pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=42, k_neighbors=k)),
        ('scaler', StandardScaler()),
        ('clf',    model)
    ])
    cv = LeaveOneOut() if use_loo else StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42)
    try:
        y_prob = cross_val_predict(pipe, X_v, y, cv=cv,
                                   method='predict_proba')[:,1]
        return evaluate(y, y_prob)
    except Exception as e:
        print(f'    Error: {e}')
        return None

def make_models():
    return {
        'SVM'      : SVC(kernel='rbf', C=1.0, gamma='scale',
                         class_weight='balanced', probability=True,
                         random_state=42),
        'XGBoost'  : xgb.XGBClassifier(n_estimators=100, max_depth=2,
                         learning_rate=0.05, subsample=0.8,
                         scale_pos_weight=5, eval_metric='logloss',
                         random_state=42),
        'LightGBM' : lgb.LGBMClassifier(n_estimators=100, max_depth=2,
                         learning_rate=0.05, class_weight='balanced',
                         random_state=42, verbose=-1),
        'ElasticNet': LogisticRegression(penalty='elasticnet', solver='saga',
                         l1_ratio=0.5, C=0.1, class_weight='balanced',
                         max_iter=2000, random_state=42),
    }

print('Models ready!')

In [ ]:
# Run all 7 experiments × 5 labels─
from sklearn.model_selection import StratifiedKFold

EXPERIMENTS = [
    ('1. Clinical only',      X_clin,    y_4tasks, False),
    ('2. Genomic only',       X_genomic, y_4tasks, False),
    ('3. Clinical+Genomic',   X_pca,     y_4tasks, False),
    ('4. Histo only (best)',  None,       y_64,     True),  # best per label
    ('5. Clinical+Histo',     None,       y_64,     True),
    ('6. Genomic+Histo',      None,       y_64,     True),
    ('7. All 3 modalities',   None,       y_64,     True),
]

all_results = []

for exp_name, X_fixed, y_df, use_loo in EXPERIMENTS:
    print(f'\n{"="*60}')
    print(f'{exp_name}  (cv={"LOO" if use_loo else "5-fold"})')
    print(f'{"="*60}')

    for label in LABEL_COLS:
        # Pick best histo for this label
        best_h = BEST_HISTO[label]

        if '4.' in exp_name:
            X = best_h
        elif '5.' in exp_name:
            X = clean_X(X_clin_64.join(best_h, how='inner'))
        elif '6.' in exp_name:
            X = clean_X(X_genomic_64.join(best_h, how='inner'))
        elif '7.' in exp_name:
            X = clean_X(X_pca_64.join(best_h, how='inner'))
        else:
            X = X_fixed

        y_lbl = y_df[label].reindex(X.index)
        pos   = int(y_lbl.fillna(0).sum())
        n     = y_lbl.notna().sum()
        print(f'\n  {label}  (pos={pos}/{n})')

        if pos < 3:
            print('    Skipping')
            continue

        for mname, model in make_models().items():
            m = run_cv(X, y_lbl, model, use_loo)
            if m:
                print(f'    {mname:<12} AUC={m["AUC"]:.3f}  '
                      f'F1={m["F1"]:.3f}  BalAcc={m["BalAcc"]:.3f}  '
                      f'MCC={m["MCC"]:.3f}')
                all_results.append({
                    'experiment': exp_name, 'label': label,
                    'model': mname, **m
                })

        # LateFusion for exp 7
        if '7.' in exp_name:
            n_c = X_clin_64.shape[1]
            n_g = X_genomic_64.shape[1]
            X_lf = clean_X(X_pca_64.join(best_h, how='inner'))
            y_arr = y_lbl.reindex(X_lf.index).fillna(0).astype(int).values
            y_prob = np.zeros(len(y_arr))
            for tr_idx, te_idx in LeaveOneOut().split(X_lf.values):
                try:
                    lf = LateFusion(n_clin=n_c, n_gen=n_g, random_state=42)
                    lf.fit(X_lf.values[tr_idx], y_arr[tr_idx])
                    y_prob[te_idx] = lf.predict_proba(
                        X_lf.values[te_idx].reshape(1,-1))[:,1]
                except: y_prob[te_idx] = 0.5
            m = evaluate(y_arr, y_prob)
            print(f'    {"LateFusion":<12} AUC={m["AUC"]:.3f}  '
                  f'F1={m["F1"]:.3f}  BalAcc={m["BalAcc"]:.3f}  '
                  f'MCC={m["MCC"]:.3f}')
            all_results.append({
                'experiment': exp_name, 'label': label,
                'model': 'LateFusion', **m
            })

results_df = pd.DataFrame(all_results)
results_df.to_csv(f'{OUT}/final_results.csv', index=False)
print('\n\nDone! Results saved.')

In [ ]:
# clean final paper table
import pandas as pd

OUT = '/kaggle/working'
results_df = pd.read_csv(f'{OUT}/final_results.csv')

print('='*72)
print('YOUR BEST RESULTS — PAPER TABLE')
print('='*72)

task_names = {
    'label_cancer_stage'     : 'Cancer Stage (Early vs Late)',
    'label_node_positive'    : 'Node Positive (Urgency)',
    'label_survival'         : 'Survival Prediction',
    'label_recurrence'       : 'Recurrence Prediction',
    'label_treatment_response': 'Treatment Response',
}

best = results_df.loc[results_df.groupby(
    ['label','experiment'])['AUC'].idxmax()]

summary = []
for label, tname in task_names.items():
    sub = best[best['label']==label].sort_values('AUC', ascending=False)
    if sub.empty: continue
    top = sub.iloc[0]
    summary.append({
        'Task'      : tname,
        'BestExp'   : top['experiment'],
        'BestModel' : top['model'],
        'AUC'       : top['AUC'],
        'F1'        : top['F1'],
        'BalAcc'    : top['BalAcc'],
        'MCC'       : top['MCC'],
    })

df_sum = pd.DataFrame(summary)
print(df_sum.to_string(index=False))
df_sum.to_csv(f'{OUT}/paper_table.csv', index=False)

# Multimodal gain
print(f'\n{"="*72}')
print('MULTIMODAL GAIN OVER BEST UNIMODAL')
print('='*72)
for label, tname in task_names.items():
    sub = results_df[results_df['label']==label]
    uni_exps = ['1. Clinical only','2. Genomic only','4. Histo only (best)']
    uni   = sub[sub['experiment'].isin(uni_exps)]['AUC'].max()
    multi = sub[sub['experiment'].str.contains('All 3')]['AUC'].max()
    if pd.isna(multi): continue
    gain = multi - uni
    flag = '✅' if gain > 0 else '❌'
    print(f'{tname:<30} UniModal={uni:.3f}  All3={multi:.3f}  '
          f'Gain={gain:+.3f}  {flag}')

In [ ]:
import cv2
import numpy as np
from PIL import Image

def is_tumor_tile(path):
    """
    Filter to high-cellularity tiles (tumor regions).
    Tumor tiles: high purple/blue content (hematoxylin staining of nuclei)
    Fat/background: white or pale pink only
    """
    img = np.array(Image.open(path).convert('RGB'))

    # Convert to HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

    # Purple/blue hue range (hematoxylin stained nuclei)
    # Hue 120-160 in HSV = blue-purple
    purple_mask = (
        (hsv[:,:,0] >= 120) & (hsv[:,:,0] <= 160) &
        (hsv[:,:,1] >= 30)   # has some saturation (not white)
    )

    # White background
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    white_pct = np.sum(gray > 220) / gray.size

    # Fat cells: large white circles — low purple, high white
    purple_pct = purple_mask.sum() / purple_mask.size

    # Good tumor tile: enough purple nuclei, not too white
    return purple_pct > 0.05 and white_pct < 0.5

# Test on your patients
CLEAN_TILES = '/kaggle/input/datasets/mdafsar07/clean-tiles/tiles1_clean'
import os
from tqdm import tqdm

tumor_manifest = []
for slide in os.listdir(CLEAN_TILES):
    slide_dir = f'{CLEAN_TILES}/{slide}'
    if not os.path.isdir(slide_dir): continue
    pid = '-'.join(slide.split('-')[:3])
    tiles = [t for t in os.listdir(slide_dir) if t.endswith('.png')]
    tumor_tiles = []
    for t in tiles:
        path = f'{slide_dir}/{t}'
        try:
            if is_tumor_tile(path):
                tumor_tiles.append(path)
        except: continue
    if tumor_tiles:
        tumor_manifest.append({
            'patient_id'  : pid,
            'total_tiles' : len(tiles),
            'tumor_tiles' : len(tumor_tiles),
            'pct_tumor'   : len(tumor_tiles)/len(tiles)*100
        })

df_tumor = pd.DataFrame(tumor_manifest)
print(f'Patients with tumor tiles: {len(df_tumor)}')
print(f'\nTumor tile stats:')
print(f'  Mean tumor tiles per patient: {df_tumor["tumor_tiles"].mean():.0f}')
print(f'  Mean % tumor: {df_tumor["pct_tumor"].mean():.1f}%')
print(f'\nWorst patients (most fat/background):')
print(df_tumor.nsmallest(5,"pct_tumor")[["patient_id","total_tiles",
                                          "tumor_tiles","pct_tumor"]].to_string())

In [ ]:
import torch
import torchvision.transforms as transforms
from efficientnet_pytorch import EfficientNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reload EfficientNet-B4
print('Loading EfficientNet-B4...')
feat_model = EfficientNet.from_pretrained('efficientnet-b4')
feat_model._fc = torch.nn.Identity()
feat_model = feat_model.to(device).eval()

# Same transform used during embedding extraction
transform = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

print('Ready!')

In [ ]:
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import torch

OUT         = '/kaggle/working'
CLEAN_TILES = '/kaggle/input/datasets/mdafsar07/clean-tiles/tiles1_clean'
BRCA_BASE   = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'

def is_tumor_tile(path):
    img  = np.array(Image.open(path).convert('RGB'))
    hsv  = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    purple_mask = ((hsv[:,:,0] >= 120) & (hsv[:,:,0] <= 160) &
                   (hsv[:,:,1] >= 30))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return (purple_mask.sum()/purple_mask.size > 0.05 and
            np.sum(gray > 220)/gray.size < 0.5)

def get_tumor_tiles(pid, max_tiles=6):
    paths = []
    for slide in os.listdir(CLEAN_TILES):
        if not slide.startswith(pid): continue
        slide_dir = f'{CLEAN_TILES}/{slide}'
        if not os.path.isdir(slide_dir): continue
        for t in os.listdir(slide_dir):
            if t.endswith('.png'):
                p = f'{slide_dir}/{t}'
                try:
                    if is_tumor_tile(p): paths.append(p)
                except: continue
    return paths[:max_tiles]

# GradCAM using feat_model already in memory
class SimpleGradCAM:
    def __init__(self, model):
        self.model = model
        self.grads = None
        self.acts  = None
        target = list(model.children())[-3]
        target.register_forward_hook(
            lambda m,i,o: setattr(self,'acts', o.detach()))
        target.register_full_backward_hook(
            lambda m,i,o: setattr(self,'grads', o[0].detach()))

    def __call__(self, t):
        self.model.eval()
        out = self.model(t)
        self.model.zero_grad()
        out.sum().backward()
        w   = self.grads.mean(dim=[2,3], keepdim=True)
        cam = (w * self.acts).sum(dim=1).squeeze()
        cam = torch.relu(cam).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

gradcam = SimpleGradCAM(feat_model)

def show_tumor_gradcam(pid, label_name, n_tiles=4):
    paths = get_tumor_tiles(pid, max_tiles=n_tiles)
    if not paths:
        print(f'  No tumor tiles for {pid}')
        return

    fig, axes = plt.subplots(3, len(paths), figsize=(4*len(paths), 12))
    fig.suptitle(f'GradCAM — {pid}  ({label_name})\n'
                 f'Tumor tiles only (purple nucleus regions)',
                 fontsize=13, fontweight='bold')

    for col, path in enumerate(paths):
        try:
            img = np.array(Image.open(path).convert('RGB'))

            # Original
            axes[0,col].imshow(img)
            axes[0,col].set_title(f'H&E Tile {col+1}', fontsize=9)
            axes[0,col].axis('off')

            # GradCAM
            t = transform(Image.open(path).convert('RGB'))
            t = t.unsqueeze(0).to(device)
            t.requires_grad_(True)
            cam = gradcam(t)

            # Heatmap only
            cam_resized = np.array(
                Image.fromarray((cam*255).astype(np.uint8)).resize(
                    (img.shape[1], img.shape[0]), Image.BILINEAR)
            ) / 255.0
            axes[1,col].imshow(cam_resized, cmap='hot', vmin=0, vmax=1)
            axes[1,col].set_title('Activation Map', fontsize=9)
            axes[1,col].axis('off')

            # Overlay
            colormap = cm.hot(cam_resized)[:,:,:3]
            overlay  = np.clip(0.5*colormap + 0.5*img/255.0, 0, 1)
            axes[2,col].imshow(overlay)
            axes[2,col].set_title('GradCAM Overlay', fontsize=9)
            axes[2,col].axis('off')

        except Exception as e:
            print(f'  Error tile {col}: {e}')
            for r in range(3): axes[r,col].axis('off')

    plt.tight_layout()
    fname = f'{OUT}/gradcam_tumor_{pid}_{label_name.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

# Load survival labels
y_surv = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv',
                     index_col='patient_id')['label_survival']
y_treat = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv',
                      index_col='patient_id')['label_treatment_response']

# Get patients with good tumor tiles (>30% tumor)
good_pats = [r['patient_id'] for _, r in
             pd.DataFrame(tumor_manifest).iterrows()
             if r['pct_tumor'] > 30 and
             not pd.isna(y_surv.get(r['patient_id'], np.nan))]

# Show 2 positive + 2 negative for survival
pos_pids = [p for p in good_pats if y_surv.get(p,np.nan)==1][:2]
neg_pids = [p for p in good_pats if y_surv.get(p,np.nan)==0][:2]

print('=== SURVIVAL — POSITIVE (Died) ===')
for pid in pos_pids:
    print(f'\n{pid}')
    show_tumor_gradcam(pid, 'Survival_Died')

print('\n=== SURVIVAL — NEGATIVE (Alive) ===')
for pid in neg_pids:
    print(f'\n{pid}')
    show_tumor_gradcam(pid, 'Survival_Alive')

# Treatment response — your best result (AUC=0.807)
pos_treat = [p for p in good_pats if y_treat.get(p,np.nan)==1][:2]
neg_treat = [p for p in good_pats if y_treat.get(p,np.nan)==0][:2]

print('\n=== TREATMENT RESPONSE — COMPLETE RESPONDERS ===')
for pid in pos_treat:
    print(f'\n{pid}')
    show_tumor_gradcam(pid, 'Treatment_CompleteResponse')

print('\n=== TREATMENT RESPONSE — NON-RESPONDERS ===')
for pid in neg_treat:
    print(f'\n{pid}')
    show_tumor_gradcam(pid, 'Treatment_NonResponse')

print('\nDone! GradCAM saved for paper figures.')

In [ ]:
val()
        try:
            out = self.model(img_tensor)
            self.model.zero_grad()
            out.sum().backward()

            # Handle different output shapes
            grads = self.grads
            acts  = self.acts

            if len(grads.shape) == 4:   # (B, C, H, W)
                w   = grads.mean(dim=[2,3], keepdim=True)
                cam = (w * acts).sum(dim=1).squeeze()
            elif len(grads.shape) == 3:  # (B, N, C) — transformer
                w   = grads.mean(dim=1)
                cam = (w.unsqueeze(1) * acts).sum(dim=-1).squeeze()
                # Reshape to square
                n = cam.shape[0]
                s = int(n**0.5)
                cam = cam[:s*s].reshape(s, s)
            else:
                return None

            cam = torch.relu(cam)
            cam = cam.cpu().numpy()
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            return cam
        except Exception as e:
            print(f'    GradCAM error: {e}')
            return None

gradcam = FixedGradCAM(feat_model)

def show_tumor_gradcam_fixed(pid, label_name, n_tiles=4):
    # Get tumor tiles
    paths = []
    for slide in os.listdir(CLEAN_TILES):
        if not slide.startswith(pid): continue
        sd = f'{CLEAN_TILES}/{slide}'
        if not os.path.isdir(sd): continue
        for t in os.listdir(sd):
            if t.endswith('.png'):
                p = f'{sd}/{t}'
                try:
                    if is_tumor_tile(p): paths.append(p)
                except: continue
    paths = paths[:n_tiles]
    if not paths:
        print(f'  No tumor tiles for {pid}')
        return

    n_cols = len(paths)
    fig, axes = plt.subplots(3, n_cols, figsize=(4*n_cols, 11))
    if n_cols == 1: axes = axes.reshape(-1,1)

    fig.suptitle(
        f'GradCAM — Patient: {pid}\n({label_name}) | Tumor tiles only',
        fontsize=13, fontweight='bold', y=1.01
    )

    row_labels = ['H&E Original', 'Activation Heatmap', 'GradCAM Overlay']
    for row, rl in enumerate(row_labels):
        axes[row,0].set_ylabel(rl, fontsize=10, fontweight='bold')

    for col, path in enumerate(paths):
        try:
            img = np.array(Image.open(path).convert('RGB'))

            # Row 0: Original H&E
            axes[0,col].imshow(img)
            axes[0,col].set_title(f'Tile {col+1}', fontsize=9)
            axes[0,col].axis('off')

            # GradCAM
            t = transform(Image.open(path).convert('RGB'))
            t = t.unsqueeze(0).to(device)
            t.requires_grad_(True)
            cam = gradcam(t)

            if cam is not None:
                cam_resized = np.array(
                    Image.fromarray((cam*255).astype(np.uint8)).resize(
                        (img.shape[1], img.shape[0]), Image.BILINEAR)
                ) / 255.0

                # Row 1: Pure heatmap
                axes[1,col].imshow(cam_resized, cmap='jet', vmin=0, vmax=1)
                axes[1,col].set_title(f'max={cam.max():.2f}', fontsize=8)
                axes[1,col].axis('off')

                # Row 2: Overlay (60% original + 40% heatmap)
                colormap = cm.jet(cam_resized)[:,:,:3]
                overlay  = np.clip(0.6*img/255.0 + 0.4*colormap, 0, 1)
                axes[2,col].imshow(overlay)
                axes[2,col].axis('off')
            else:
                # Just show original if GradCAM failed
                for row in [1,2]:
                    axes[row,col].imshow(img, alpha=0.5)
                    axes[row,col].set_title('GradCAM N/A', fontsize=8)
                    axes[row,col].axis('off')

        except Exception as e:
            print(f'  Col {col} error: {e}')
            for row in range(3): axes[row,col].axis('off')

    plt.tight_layout()
    fname = f'{OUT}/gradcam_{pid}_{label_name.replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

# Load labels
import pandas as pd
y_all  = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')
y_surv = y_all['label_survival']
y_treat= y_all['label_treatment_response']

# Get good patients (>30% tumor tiles, skip TCGA-BH-A1EY which has almost none)
skip = {'TCGA-BH-A1EY', 'TCGA-AC-A3QQ'}
df_tumor = pd.DataFrame(tumor_manifest)
good = set(df_tumor[df_tumor['pct_tumor']>30]['patient_id']) - skip

# Survival: 2 died + 2 alive
pos_s = [p for p in good if y_surv.get(p,float('nan'))==1][:2]
neg_s = [p for p in good if y_surv.get(p,float('nan'))==0][:2]

# Treatment: complete responders + non-responders
pos_t = [p for p in good if y_treat.get(p,float('nan'))==1][:2]
neg_t = [p for p in good if y_treat.get(p,float('nan'))==0][:2]

print('=== SURVIVAL PREDICTION ===')
print(f'Positive (Died)  : {pos_s}')
print(f'Negative (Alive) : {neg_s}')
print(f'\n=== TREATMENT RESPONSE ===')
print(f'Positive (Complete): {pos_t}')
print(f'Negative (Other)   : {neg_t}')

for pid in pos_s:
    show_tumor_gradcam_fixed(pid, 'Survival-Died')
for pid in neg_s:
    show_tumor_gradcam_fixed(pid, 'Survival-Alive')
for pid in pos_t:
    show_tumor_gradcam_fixed(pid, 'Treatment-CompleteResponse')
for pid in neg_t:
    show_tumor_gradcam_fixed(pid, 'Treatment-NonResponse')

print('\nAll GradCAM saved!')

In [22]:
!pip install -q efficientnet_pytorch timm lightgbm imbalanced-learn transformers

In [23]:
import os, glob, warnings, pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

TILES_ROOT = '/kaggle/input/datasets/mdafsar07/final-histo/Final_Clean_Tiles'
BRCA_BASE  = '/kaggle/input/datasets/jannatulferdoushritu/brca-pre-processed-dataset'
OUT        = '/kaggle/working'

TUMOR_ONLY       = False  # True = primary-tumor slides only (sample type 01); False = all slides
MAX_TILES_SAMPLE = 150
MAX_TILES_EMB    = 100

LABEL_COLS = ['label_cancer_stage','label_node_positive',
              'label_survival','label_recurrence','label_treatment_response']

LEAK_COLS = [
    'fu.follow_ups.days_to_recurrence','extra_days_to_recurrence',
    'extra_days_to_progression','demographic.days_to_death',
    'diagnoses.days_to_last_follow_up','fu.follow_ups.days_to_follow_up',
    'disease_response_wt-with tumor','disease_response_tf-tumor free',
    'disease_response_unknown','disease_response_wt-with tumor.1',
    'disease_response_tf-tumor free.1','disease_response_unknown.1',
    'fu.follow_ups.disease_response','extra_disease_response_tf-tumor free',
    'extra_disease_response_wt-with tumor','extra_disease_response_unknown',
    'progression_or_recurrence_yes','progression_or_recurrence_unknown',
    'ord_ajcc_pathologic_stage','ord_ajcc_pathologic_n',
    'ord_ajcc_pathologic_m','ord_ajcc_pathologic_t','ord_tumor_grade',
    'progression_or_recurrence_type_distant',
    'progression_or_recurrence_type_locoregional',
    'progression_or_recurrence_type_unknown',
]

def clean_X(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object or df[col].dtype == bool:
            df[col] = df[col].map(
                {'True':1,'False':0,True:1,False:0,
                 'true':1,'false':0,'yes':1,'no':0}
            ).fillna(df[col])
            df[col] = pd.to_numeric(df[col], errors='coerce')
    all_nan = df.columns[df.isna().all()].tolist()
    if all_nan: df = df.drop(columns=all_nan)
    imp = SimpleImputer(strategy='median')
    return pd.DataFrame(imp.fit_transform(df), index=df.index, columns=df.columns)

def remove_leaks(df):
    return df.drop(columns=[c for c in LEAK_COLS if c in df.columns])

X_clin_raw = pd.read_csv(f'{BRCA_BASE}/X_clinical.csv', index_col='patient_id')
X_clin     = clean_X(remove_leaks(X_clin_raw))
X_pca      = clean_X(remove_leaks(
    pd.read_csv(f'{BRCA_BASE}/X_pca_v2.csv', index_col='patient_id')))
X_genomic  = clean_X(X_pca.filter(like='PC'))
y_old      = pd.read_csv(f'{BRCA_BASE}/y_labels_v2.csv', index_col='patient_id')

stage   = X_clin_raw['ord_ajcc_pathologic_stage']
n_stage = X_clin_raw['ord_ajcc_pathologic_n']
label_stage = pd.Series(np.nan, index=X_clin_raw.index)
label_stage[stage >= 8] = 1
label_stage[(stage >= 1) & (stage <= 6)] = 0
label_node = pd.Series(np.nan, index=X_clin_raw.index)
label_node[n_stage >= 6] = 1
label_node[(n_stage >= 2) & (n_stage <= 5)] = 0
y_4tasks = pd.DataFrame({
    'label_cancer_stage'      : label_stage,
    'label_node_positive'     : label_node,
    'label_survival'          : y_old['label_survival'],
    'label_recurrence'        : y_old['label_recurrence'],
    'label_treatment_response': y_old['label_treatment_response'],
}, index=X_clin_raw.index)
y_4tasks.to_csv(f'{OUT}/y_labels_4tasks.csv')

print(f'X_clin    : {X_clin.shape}')
print(f'X_genomic : {X_genomic.shape}')
print(f'X_pca     : {X_pca.shape}')
print(f'y_4tasks  : {y_4tasks.shape}')
print('\nLabels (all patients):')
for c in LABEL_COLS:
    n = y_4tasks[c].notna().sum(); p = int(y_4tasks[c].fillna(0).sum())
    print(f'  {c:<28} n={n:<5} pos={p:<4} rate={p/n:.1%}')
print(f'\nTUMOR_ONLY = {TUMOR_ONLY}')

X_clin    : (1095, 103)
X_genomic : (1095, 100)
X_pca     : (1095, 204)
y_4tasks  : (1095, 5)

Labels (all patients):
  label_cancer_stage           n=994   pos=246  rate=24.7%
  label_node_positive          n=985   pos=496  rate=50.4%
  label_survival               n=1095  pos=152  rate=13.9%
  label_recurrence             n=1095  pos=120  rate=11.0%
  label_treatment_response     n=1095  pos=124  rate=11.3%

TUMOR_ONLY = False


In [24]:
slides = sorted(s for s in os.listdir(TILES_ROOT)
                if os.path.isdir(f'{TILES_ROOT}/{s}'))
slide_df = pd.DataFrame({
    'slide'     : slides,
    'patient_id': ['-'.join(s.split('-')[:3]) for s in slides],
    'sample'    : [s.split('-')[3][:2] for s in slides],   # 01 tumor, 11 normal, 06 metastatic
})
tumor_pids = set(slide_df.loc[slide_df['sample']=='01', 'patient_id'])
all_pids   = set(slide_df['patient_id'])

print(f'Slide folders           : {len(slide_df)}')
print(f'Patients (all slides)   : {len(all_pids)}')
print(f'Sample types            : {slide_df["sample"].value_counts().to_dict()}')
print(f'Patients with tumor 01  : {len(tumor_pids)}')
print(f'Normal-only patients    : {len(all_pids - tumor_pids)}')

if TUMOR_ONLY:
    slide_df = slide_df[slide_df['sample']=='01']
keep_pids = set(slide_df['patient_id'])

cohort_ids = sorted(keep_pids & set(X_clin.index) & set(X_genomic.index))
slide_df   = slide_df[slide_df['patient_id'].isin(cohort_ids)]
print(f'\nSlides used             : {len(slide_df)}')
print(f'COHORT (all 3 modalities): {len(cohort_ids)} patients')

records = []
for _, r in slide_df.iterrows():
    sd = f'{TILES_ROOT}/{r["slide"]}'
    for t in os.listdir(sd):
        if t.endswith('.png'):
            records.append({'patient_id': r['patient_id'],
                            'slide'     : r['slide'],
                            'tile_path' : f'{sd}/{t}'})
all_tiles = pd.DataFrame(records)
manifest  = (all_tiles.sample(frac=1, random_state=42)
                      .groupby('patient_id').head(MAX_TILES_SAMPLE)
                      .reset_index(drop=True))
tile_dict = {pid: g['tile_path'].tolist() for pid, g in manifest.groupby('patient_id')}
manifest.to_csv(f'{OUT}/tile_manifest_final.csv', index=False)

n_per = all_tiles.groupby('patient_id').size()
print(f'\nTotal tiles available   : {len(all_tiles):,}')
print(f'Tiles per patient       : min={n_per.min()}  mean={n_per.mean():.0f}  max={n_per.max()}')
print(f'Tiles sampled           : {len(manifest):,}  ({len(tile_dict)} patients)')

y_N = y_4tasks.loc[cohort_ids]
print(f'\nLabel distribution — {len(cohort_ids)} patients:')
print(f'  {"label":<28} {"n":>4} {"pos":>4} {"rate":>7}')
for c in LABEL_COLS:
    n = int(y_N[c].notna().sum()); p = int(y_N[c].fillna(0).sum())
    flag = '   ⚠ <3 positives, will be skipped' if p < 3 else ''
    print(f'  {c:<28} {n:>4} {p:>4} {p/n:>7.1%}{flag}')

Slide folders           : 140
Patients (all slides)   : 118
Sample types            : {'01': 101, '11': 38, '06': 1}
Patients with tumor 01  : 93
Normal-only patients    : 25

Slides used             : 140
COHORT (all 3 modalities): 118 patients

Total tiles available   : 50,718
Tiles per patient       : min=58  mean=430  max=2432
Tiles sampled           : 17,196  (118 patients)

Label distribution — 118 patients:
  label                           n  pos    rate
  label_cancer_stage            108   32   29.6%
  label_node_positive           109   57   52.3%
  label_survival                118   25   21.2%
  label_recurrence              118   21   17.8%
  label_treatment_response      118   10    8.5%


In [25]:
# Reads each tile once and feeds BOTH models. Resumable: re-run if the session dies.
# (If you change TUMOR_ONLY, delete /kaggle/working/emb_checkpoint.pkl first.)
import cv2, torch, timm
import torchvision.transforms as transforms
from concurrent.futures import ThreadPoolExecutor
from efficientnet_pytorch import EfficientNet as EfficientNetModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

eff_net = EfficientNetModel.from_pretrained('efficientnet-b4')
eff_net._fc = torch.nn.Identity()
eff_net = eff_net.to(device).eval()
vit = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0)
vit = vit.to(device).eval()

eff_transform = transforms.Compose([
    transforms.Resize((380, 380)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
vit_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])

def load_tile(p, check_tissue=True):
    try:
        img = Image.open(p).convert('RGB')
        if check_tissue:
            gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
            if np.sum(gray > 220)/gray.size >= 0.4:
                return None
        return eff_transform(img), vit_transform(img)
    except Exception:
        return None

def embed(model, x, bs=25):
    outs = []
    for i in range(0, len(x), bs):
        with torch.no_grad():
            e = model(x[i:i+bs].to(device))
            if e.ndim > 2: e = e.squeeze(-1).squeeze(-1)
        outs.append(e.cpu())
    return torch.cat(outs).mean(dim=0).numpy()

ckpt_path = f'{OUT}/emb_checkpoint.pkl'
ckpt = pickle.load(open(ckpt_path,'rb')) if os.path.exists(ckpt_path) else {'eff':{}, 'vit':{}}
eff_embs, vit_embs = ckpt['eff'], ckpt['vit']
print(f'Resuming: {len(eff_embs)} patients already done')

pool = ThreadPoolExecutor(max_workers=4)
todo = [p for p in tile_dict if p not in eff_embs]
for k, pid in enumerate(tqdm(todo, desc='Embedding')):
    res, paths = [], tile_dict[pid]
    for i in range(0, len(paths), 50):            # load in chunks until 100 tissue tiles
        res += [r for r in pool.map(load_tile, paths[i:i+50]) if r is not None]
        if len(res) >= MAX_TILES_EMB: break
    res = res[:MAX_TILES_EMB]
    if not res:                                   # no tissue tile found → use anything
        res = [r for r in pool.map(lambda p: load_tile(p, False),
                                   tile_dict[pid][:MAX_TILES_EMB]) if r is not None]
    if not res: continue
    eff_embs[pid] = embed(eff_net, torch.stack([r[0] for r in res]))
    vit_embs[pid] = embed(vit,     torch.stack([r[1] for r in res]))
    if (k+1) % 10 == 0:
        pickle.dump({'eff':eff_embs,'vit':vit_embs}, open(ckpt_path,'wb'))
pickle.dump({'eff':eff_embs,'vit':vit_embs}, open(ckpt_path,'wb'))
del eff_net, vit
if device.type == 'cuda': torch.cuda.empty_cache()

def make_pca(emb_dict, n_pcs=50, prefix='PC'):
    raw = pd.DataFrame(emb_dict).T
    raw.index.name = 'patient_id'
    N  = min(n_pcs, len(raw)-1)
    pc = PCA(n_components=N, random_state=42)
    df = pd.DataFrame(pc.fit_transform(StandardScaler().fit_transform(raw)),
                      index=raw.index, columns=[f'{prefix}{i+1}' for i in range(N)])
    return df, pc.explained_variance_ratio_.sum()

eff_pca, v1 = make_pca(eff_embs, 50, 'eff_PC')
vit_pca, v2 = make_pca(vit_embs, 50, 'vit_PC')
eff_pca.to_csv(f'{OUT}/X_histo_eff.csv')
vit_pca.to_csv(f'{OUT}/X_histo_vit.csv')
print(f'\nEfficientNet-B4 PCA : {eff_pca.shape}  var={v1:.1%}')
print(f'ViT-Small PCA       : {vit_pca.shape}  var={v2:.1%}')
print(f'Patients embedded   : {len(eff_embs)} / {len(tile_dict)}')

Device: cuda
Loaded pretrained weights for efficientnet-b4


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Resuming: 0 patients already done


Embedding: 100%|██████████| 118/118 [09:30<00:00,  4.84s/it]



EfficientNet-B4 PCA : (118, 50)  var=95.1%
ViT-Small PCA       : (118, 50)  var=97.2%
Patients embedded   : 118 / 118


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             balanced_accuracy_score, matthews_corrcoef)
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

eff_pca = pd.read_csv(f'{OUT}/X_histo_eff.csv', index_col='patient_id')
vit_pca = pd.read_csv(f'{OUT}/X_histo_vit.csv', index_col='patient_id')
y_4tasks = pd.read_csv(f'{OUT}/y_labels_4tasks.csv', index_col='patient_id')
LOO = LeaveOneOut()

def quick_auc(X, y_series):
    y = y_series.reindex(X.index).dropna().astype(int)
    X_v = X.loc[y.index]
    if y.sum() < 3: return np.nan
    pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=42, k_neighbors=min(3, int(y.sum())-1))),
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(penalty='elasticnet', solver='saga',
                       l1_ratio=0.5, C=0.1, class_weight='balanced',
                       max_iter=2000, random_state=42))])
    try:
        prob = cross_val_predict(pipe, X_v, y, cv=LOO, method='predict_proba')[:,1]
        return round(roc_auc_score(y, prob), 3)
    except Exception:
        return np.nan

print('='*62)
print(f'HISTO FEATURE COMPARISON (LOO, ElasticNet, {len(eff_pca)} patients)')
print('='*62)
print(f'{"Label":<28} {"EfficientNet":>13} {"ViT-Small":>10} {"Best":>6}')
BEST_HISTO, BEST_NAME, rows = {}, {}, []
for label in LABEL_COLS:
    a_eff = quick_auc(eff_pca, y_4tasks[label])
    a_vit = quick_auc(vit_pca, y_4tasks[label])
    use_vit = (not np.isnan(a_vit)) and (np.isnan(a_eff) or a_vit > a_eff)
    BEST_HISTO[label] = vit_pca if use_vit else eff_pca
    BEST_NAME[label]  = 'ViT' if use_vit else 'Eff'
    print(f'{label:<28} {a_eff:>13} {a_vit:>10} {BEST_NAME[label]:>6}')
    rows.append({'label':label,'EfficientNet':a_eff,'ViT':a_vit,'best':BEST_NAME[label]})
pd.DataFrame(rows).to_csv(f'{OUT}/feature_comparison.csv', index=False)

histo_ensemble = eff_pca.join(vit_pca, how='inner')
pc = PCA(n_components=50, random_state=42)
histo_combined = pd.DataFrame(
    pc.fit_transform(StandardScaler().fit_transform(histo_ensemble)),
    index=histo_ensemble.index, columns=[f'ens_PC{i+1}' for i in range(50)])
histo_combined.to_csv(f'{OUT}/X_histo_ensemble.csv')
print(f'\nEnsemble histo PCA: {histo_combined.shape}  var={pc.explained_variance_ratio_.sum():.1%}')

HISTO FEATURE COMPARISON (LOO, ElasticNet, 118 patients)
Label                         EfficientNet  ViT-Small   Best
label_cancer_stage                   0.442      0.561    ViT
label_node_positive                  0.528      0.455    Eff
label_survival                       0.633      0.686    ViT
label_recurrence                      0.48      0.605    ViT
label_treatment_response             0.504      0.639    ViT

Ensemble histo PCA: (118, 50)  var=91.1%


In [27]:
idx_N        = list(eff_pca.index)
X_clin_N     = X_clin.loc[idx_N]
X_genomic_N  = X_genomic.loc[idx_N]
X_pca_N      = X_pca.loc[idx_N]
y_N          = y_4tasks.loc[idx_N]
print(f'Cohort: {len(idx_N)} patients')
print(f'  X_clin_N {X_clin_N.shape} | X_genomic_N {X_genomic_N.shape} | X_pca_N {X_pca_N.shape}')

class LateFusion(BaseEstimator, ClassifierMixin):
    """3 experts (clinical, genomic, histo) + logistic meta-learner."""
    def __init__(self, n_clin, n_gen, random_state=42):
        self.n_clin = n_clin
        self.n_gen  = n_gen
        self.random_state = random_state
        self.classes_ = np.array([0,1])
    def _split(self, X):
        return (X[:, :self.n_clin],
                X[:, self.n_clin:self.n_clin+self.n_gen],
                X[:, self.n_clin+self.n_gen:])
    def _expert(self, seed):
        return ImbPipeline([
            ('smote',  SMOTE(random_state=seed, k_neighbors=2)),
            ('scaler', StandardScaler()),
            ('pca',    PCA(n_components=10, random_state=seed)),
            ('clf',    LogisticRegression(penalty='elasticnet', solver='saga',
                           l1_ratio=0.5, C=0.3, class_weight='balanced',
                           max_iter=2000, random_state=seed))])
    def fit(self, X, y):
        X_c, X_g, X_h = self._split(X)
        self.e_c_ = self._expert(self.random_state)
        self.e_g_ = self._expert(self.random_state+1)
        self.e_h_ = self._expert(self.random_state+2)
        self.e_c_.fit(X_c, y); self.e_g_.fit(X_g, y); self.e_h_.fit(X_h, y)
        p = np.column_stack([self.e_c_.predict_proba(X_c)[:,1],
                             self.e_g_.predict_proba(X_g)[:,1],
                             self.e_h_.predict_proba(X_h)[:,1]])
        self.meta_ = LogisticRegression(C=1.0, class_weight='balanced',
                                        max_iter=1000, random_state=self.random_state)
        self.meta_.fit(p, y)
        return self
    def predict_proba(self, X):
        X_c, X_g, X_h = self._split(X)
        p = np.column_stack([self.e_c_.predict_proba(X_c)[:,1],
                             self.e_g_.predict_proba(X_g)[:,1],
                             self.e_h_.predict_proba(X_h)[:,1]])
        return self.meta_.predict_proba(p)
    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

def evaluate(y_true, y_prob):
    y_true = np.asarray(y_true); y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
    try:    auc = roc_auc_score(y_true, y_prob)
    except Exception: auc = 0.5
    return {'AUC'     : round(auc,3),
            'F1'      : round(f1_score(y_true,y_pred,pos_label=1,zero_division=0),3),
            'F1_Neg'  : round(f1_score(y_true,y_pred,pos_label=0,zero_division=0),3),
            'Prec_Pos': round(precision_score(y_true,y_pred,pos_label=1,zero_division=0),3),
            'Rec_Pos' : round(recall_score(y_true,y_pred,pos_label=1,zero_division=0),3),
            'BalAcc'  : round(balanced_accuracy_score(y_true,y_pred),3),
            'MCC'     : round(matthews_corrcoef(y_true,y_pred),3)}

def run_cv(X, y_series, model, use_loo=False):
    y = y_series.reindex(X.index).dropna().astype(int)
    X_v = X.loc[y.index]
    if y.nunique() < 2 or y.sum() < 3: return None
    pipe = ImbPipeline([
        ('smote',  SMOTE(random_state=42, k_neighbors=min(3, int(y.sum())-1))),
        ('scaler', StandardScaler()),
        ('clf',    model)])
    cv = LeaveOneOut() if use_loo else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    try:
        y_prob = cross_val_predict(pipe, X_v, y, cv=cv, method='predict_proba')[:,1]
        return evaluate(y, y_prob)
    except Exception as e:
        print(f'    Error: {e}')
        return None

def make_models():
    return {
        'SVM'       : SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
                          probability=True, random_state=42),
        'XGBoost'   : xgb.XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05,
                          subsample=0.8, scale_pos_weight=5, eval_metric='logloss',
                          random_state=42),
        'LightGBM'  : lgb.LGBMClassifier(n_estimators=100, max_depth=2, learning_rate=0.05,
                          class_weight='balanced', random_state=42, verbose=-1),
        'ElasticNet': LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                          C=0.1, class_weight='balanced', max_iter=2000, random_state=42),
    }
print('LateFusion, evaluate, run_cv, make_models ready')

Cohort: 118 patients
  X_clin_N (118, 103) | X_genomic_N (118, 100) | X_pca_N (118, 204)
LateFusion, evaluate, run_cv, make_models ready


In [ ]:
EXPERIMENTS = [
    ('1. Clinical only',     'full'),
    ('2. Genomic only',      'full'),
    ('3. Clinical+Genomic',  'full'),
    ('4. Histo only (best)', 'cohort'),
    ('5. Clinical+Histo',    'cohort'),
    ('6. Genomic+Histo',     'cohort'),
    ('7. All 3 modalities',  'cohort'),
]
FULL_X = {'1.': X_clin, '2.': X_genomic, '3.': X_pca}

all_results = []
for exp_name, scope in EXPERIMENTS:
    print(f'\n{"="*62}\n{exp_name}  ({"1095 patients, 5-fold" if scope=="full" else f"{len(idx_N)} patients, LOO"})\n{"="*62}')
    for label in LABEL_COLS:
        h = BEST_HISTO[label]
        if   exp_name[:2] in FULL_X:  X = FULL_X[exp_name[:2]]
        elif exp_name.startswith('4'): X = h
        elif exp_name.startswith('5'): X = clean_X(X_clin_N.join(h, how='inner'))
        elif exp_name.startswith('6'): X = clean_X(X_genomic_N.join(h, how='inner'))
        else:                          X = clean_X(X_pca_N.join(h, how='inner'))
        use_loo = (scope == 'cohort')
        y_lbl = y_4tasks[label].reindex(X.index)
        n, pos = int(y_lbl.notna().sum()), int(y_lbl.fillna(0).sum())
        print(f'\n  {label}  (pos={pos}/{n})' + (f'  [histo: {BEST_NAME[label]}]' if scope=="cohort" else ''))
        if pos < 3:
            print('    Skipping — fewer than 3 positives'); continue
        for mname, model in make_models().items():
            m = run_cv(X, y_lbl, model, use_loo)
            if m:
                print(f'    {mname:<11} AUC={m["AUC"]:.3f} F1={m["F1"]:.3f} BalAcc={m["BalAcc"]:.3f} MCC={m["MCC"]:.3f}')
                all_results.append({'experiment':exp_name,'label':label,'model':mname,'n':n,**m})

        if exp_name.startswith('7'):                 # LateFusion: columns = [clin | genomic | histo]
            X_lf = pd.concat([X_clin_N, X_genomic_N, h.loc[idx_N]], axis=1)
            yy   = y_lbl.reindex(X_lf.index).dropna().astype(int)
            Xa, ya = X_lf.loc[yy.index].values, yy.values
            prob = np.full(len(ya), 0.5)
            for tr, te in LeaveOneOut().split(Xa):
                if len(np.unique(ya[tr])) < 2: continue
                try:
                    lf = LateFusion(X_clin_N.shape[1], X_genomic_N.shape[1], 42).fit(Xa[tr], ya[tr])
                    prob[te] = lf.predict_proba(Xa[te])[:,1]
                except Exception:
                    pass
            m = evaluate(ya, prob)
            print(f'    {"LateFusion":<11} AUC={m["AUC"]:.3f} F1={m["F1"]:.3f} BalAcc={m["BalAcc"]:.3f} MCC={m["MCC"]:.3f}')
            all_results.append({'experiment':exp_name,'label':label,'model':'LateFusion','n':len(ya),**m})
    pd.DataFrame(all_results).to_csv(f'{OUT}/final_results.csv', index=False)   # saved after every experiment

results_df = pd.DataFrame(all_results)
print('\n\nDone — saved final_results.csv')

In [ ]:
results_df = pd.read_csv(f'{OUT}/final_results.csv')
task_names = {'label_cancer_stage':'Cancer Stage (Early vs Late)',
              'label_node_positive':'Node Positive',
              'label_survival':'Survival',
              'label_recurrence':'Recurrence',
              'label_treatment_response':'Treatment Response'}

best = results_df.loc[results_df.groupby(['label','experiment'])['AUC'].idxmax()]
for label, tname in task_names.items():
    sub = best[best['label']==label].sort_values('experiment')
    if sub.empty: continue
    print(f'\n{tname}')
    print(f'  {"Experiment":<22}{"Model":<12}{"n":>5}{"AUC":>7}{"F1+":>7}{"F1-":>7}{"Prec+":>7}{"Rec+":>7}{"BalAcc":>8}{"MCC":>7}')
    print('  '+'-'*84)
    for _, r in sub.iterrows():
        star = ' ★' if r['AUC']==sub['AUC'].max() else ''
        print(f'  {r.experiment:<22}{r.model:<12}{int(r.n):>5}{r.AUC:>7.3f}{r.F1:>7.3f}{r.F1_Neg:>7.3f}'
              f'{r.Prec_Pos:>7.3f}{r.Rec_Pos:>7.3f}{r.BalAcc:>8.3f}{r.MCC:>7.3f}{star}')

print('\n'+'='*70+'\nMULTIMODAL GAIN OVER BEST UNIMODAL\n'+'='*70)
uni_exps = ['1. Clinical only','2. Genomic only','4. Histo only (best)']
summary = []
for label, tname in task_names.items():
    sub = results_df[results_df['label']==label]
    uni, multi = sub[sub.experiment.isin(uni_exps)]['AUC'].max(), sub[sub.experiment.str.contains('All 3')]['AUC'].max()
    if pd.isna(multi): continue
    top = best[best['label']==label].sort_values('AUC', ascending=False).iloc[0]
    print(f'{tname:<30} Unimodal={uni:.3f}  All3={multi:.3f}  Gain={multi-uni:+.3f} {"✅" if multi>uni else "❌"}')
    summary.append({'Task':tname,'BestExp':top.experiment,'BestModel':top.model,'AUC':top.AUC,
                    'F1':top.F1,'BalAcc':top.BalAcc,'MCC':top.MCC,'UniAUC':uni,'All3AUC':multi})
pd.DataFrame(summary).to_csv(f'{OUT}/paper_table.csv', index=False)
print('\nSaved paper_table.csv and final_results.csv')

In [ ]:
rows = []
for exp_name, X in [('1b. Clinical only (118)',      X_clin_N),
                    ('2b. Genomic only (118)',       X_genomic_N),
                    ('3b. Clinical+Genomic (118)',   X_pca_N)]:
    print(f'\n{exp_name}')
    for label in LABEL_COLS:
        y_lbl = y_4tasks[label].reindex(X.index)
        if int(y_lbl.fillna(0).sum()) < 3: continue
        for mname, model in make_models().items():
            m = run_cv(X, y_lbl, model, use_loo=True)
            if m:
                rows.append({'experiment':exp_name,'label':label,'model':mname,
                             'n':int(y_lbl.notna().sum()),**m})
        print(f'  {label:<26} done')
same118 = pd.DataFrame(rows)
same118.to_csv(f'{OUT}/results_same118_baselines.csv', index=False)

allr = pd.concat([pd.read_csv(f'{OUT}/final_results.csv')
                    .query("experiment.str.startswith(('4','5','6','7'))", engine='python'),
                  same118], ignore_index=True)
order = ['1b. Clinical only (118)','2b. Genomic only (118)','3b. Clinical+Genomic (118)',
         '4. Histo only (best)','5. Clinical+Histo','6. Genomic+Histo','7. All 3 modalities']
std = allr[allr.model != 'LateFusion']

print('\n' + '='*86)
print('SAME 118 PATIENTS, LOO — AUC: best model | mean of 4 models')
print('='*86)
for label in LABEL_COLS:
    print(f'\n{label}')
    for e in order:
        s = std[(std.label==label) & (std.experiment==e)]
        if s.empty: continue
        print(f'  {e:<28} best={s.AUC.max():.3f} ({s.loc[s.AUC.idxmax(),"model"]:<10}) mean={s.AUC.mean():.3f}')
    uni  = std[(std.label==label) & std.experiment.isin(order[:2]+[order[3]])].groupby('experiment').AUC.mean().max()
    all3 = std[(std.label==label) & (std.experiment==order[6])].AUC.mean()
    print(f'  → mean-AUC gain, All-3 vs best unimodal: {all3-uni:+.3f}')

In [ ]:
same118 = pd.read_csv(f'{OUT}/results_same118_baselines.csv')
main    = pd.read_csv(f'{OUT}/final_results.csv')
main    = main[main['experiment'].str[0].isin(['4','5','6','7'])]
allr    = pd.concat([main, same118], ignore_index=True)

order = ['1b. Clinical only (118)','2b. Genomic only (118)','3b. Clinical+Genomic (118)',
         '4. Histo only (best)','5. Clinical+Histo','6. Genomic+Histo','7. All 3 modalities']
std = allr[allr.model != 'LateFusion']

print('='*86)
print('SAME 118 PATIENTS, LOO — AUC: best model | mean of 4 models')
print('='*86)
for label in LABEL_COLS:
    print(f'\n{label}')
    for e in order:
        s = std[(std.label==label) & (std.experiment==e)]
        if s.empty: continue
        print(f'  {e:<28} best={s.AUC.max():.3f} ({s.loc[s.AUC.idxmax(),"model"]:<10}) mean={s.AUC.mean():.3f}')
    uni  = std[(std.label==label) & std.experiment.isin([order[0],order[1],order[3]])].groupby('experiment').AUC.mean().max()
    all3 = std[(std.label==label) & (std.experiment==order[6])].AUC.mean()
    print(f'  → mean-AUC gain, All-3 vs best unimodal: {all3-uni:+.3f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def is_tumor_tile(path):
    try:
        img = np.array(Image.open(path).convert('RGB'))
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        purple = (hsv[:,:,0]>=120)&(hsv[:,:,0]<=160)&(hsv[:,:,1]>=30)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return purple.mean() > 0.05 and (gray > 220).mean() < 0.5
    except Exception:
        return False

# Only primary-tumor slides (sample 01), using the sampled tiles in `manifest`
mt = manifest[manifest['slide'].str.split('-').str[3].str[:2] == '01'].copy()
with ThreadPoolExecutor(max_workers=4) as ex:
    mt['is_tumor'] = list(tqdm(ex.map(is_tumor_tile, mt['tile_path']), total=len(mt)))
tumor_tiles = {p: g.loc[g.is_tumor,'tile_path'].tolist() for p, g in mt.groupby('patient_id')}
pct_tumor   = mt.groupby('patient_id')['is_tumor'].mean()*100
print(f'Patients with tumor-slide tiles: {len(pct_tumor)} | mean % tumor tiles: {pct_tumor.mean():.1f}')

# EfficientNet-B4 + GradCAM on the last conv block
feat_model = EfficientNetModel.from_pretrained('efficientnet-b4')
feat_model._fc = torch.nn.Identity()
feat_model = feat_model.to(device).eval()

class GradCAM:
    def __init__(self, model):
        self.model, self.acts, self.grads = model, None, None
        def fwd(m, i, o):
            self.acts = o
            o.register_hook(lambda g: setattr(self, 'grads', g))
        model._blocks[-1].register_forward_hook(fwd)
    def __call__(self, x):
        self.model.zero_grad()
        self.model(x).sum().backward()
        w   = self.grads.mean(dim=[2,3], keepdim=True)
        cam = torch.relu((w*self.acts).sum(dim=1)).squeeze().detach().cpu().numpy()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-8)
gradcam = GradCAM(feat_model)

def show_gradcam(pid, tag, n_tiles=4):
    paths = tumor_tiles.get(pid, [])[:n_tiles]
    if not paths: print(f'  no tumor tiles for {pid}'); return
    fig, axes = plt.subplots(3, len(paths), figsize=(4*len(paths), 11), squeeze=False)
    fig.suptitle(f'GradCAM — {pid} ({tag})', fontsize=13, fontweight='bold')
    for c, p in enumerate(paths):
        img = np.array(Image.open(p).convert('RGB'))
        x   = eff_transform(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
        cam = gradcam(x)
        cam = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(
              (img.shape[1], img.shape[0]), Image.BILINEAR))/255.0
        axes[0,c].imshow(img);                          axes[0,c].set_title(f'Tile {c+1}', fontsize=9)
        axes[1,c].imshow(cam, cmap='jet', vmin=0, vmax=1)
        axes[2,c].imshow(np.clip(0.6*img/255.0 + 0.4*cm.jet(cam)[:,:,:3], 0, 1))
        for r in range(3): axes[r,c].axis('off')
    for r, t in enumerate(['H&E','Activation','Overlay']): axes[r,0].set_ylabel(t, fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{OUT}/gradcam_{pid}_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()

good = set(pct_tumor[pct_tumor > 30].index)
def pick(label, val, k=2):
    s = y_4tasks[label]
    return [p for p in sorted(good) if p in s.index and s[p]==val][:k]

for pid in pick('label_survival', 1):              show_gradcam(pid, 'Survival-Died')
for pid in pick('label_survival', 0):              show_gradcam(pid, 'Survival-Alive')
for pid in pick('label_treatment_response', 1):    show_gradcam(pid, 'Treatment-CompleteResponse')
for pid in pick('label_treatment_response', 0):    show_gradcam(pid, 'Treatment-NonResponse')

In [ ]:
from transformers import CLIPModel, CLIPProcessor

print('Loading PLIP...')
plip_model = CLIPModel.from_pretrained("vinid/plip", attn_implementation="eager").to(device).eval()
plip_processor = CLIPProcessor.from_pretrained("vinid/plip")
plip_vis = plip_model.vision_model

def load_pil(p):
    try:
        img = Image.open(p).convert('RGB')
        gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
        return None if (gray > 220).mean() >= 0.4 else img      # same tissue rule as Cell 3
    except Exception:
        return None

def plip_embed(imgs, bs=32):
    outs = []
    for i in range(0, len(imgs), bs):
        inp = plip_processor(images=imgs[i:i+bs], return_tensors='pt')
        with torch.no_grad():
            f = plip_vis(pixel_values=inp['pixel_values'].to(device)).pooler_output
            f = f / f.norm(dim=-1, keepdim=True)
        outs.append(f.cpu())
    return torch.cat(outs).mean(dim=0).numpy()

plip_ckpt = f'{OUT}/plip_embs.pkl'
plip_embs = pickle.load(open(plip_ckpt,'rb')) if os.path.exists(plip_ckpt) else {}
pool = ThreadPoolExecutor(max_workers=4)
for k, pid in enumerate(tqdm([p for p in tile_dict if p not in plip_embs], desc='PLIP')):
    imgs, paths = [], tile_dict[pid]
    for i in range(0, len(paths), 50):
        imgs += [m for m in pool.map(load_pil, paths[i:i+50]) if m is not None]
        if len(imgs) >= MAX_TILES_EMB: break
    imgs = imgs[:MAX_TILES_EMB] or [Image.open(p).convert('RGB') for p in paths[:MAX_TILES_EMB]]
    plip_embs[pid] = plip_embed(imgs)
    if (k+1) % 10 == 0: pickle.dump(plip_embs, open(plip_ckpt,'wb'))
pickle.dump(plip_embs, open(plip_ckpt,'wb'))

plip_pca, v = make_pca(plip_embs, 50, 'plip_PC')
plip_pca.to_csv(f'{OUT}/X_histo_plip.csv')
print(f'PLIP PCA: {plip_pca.shape}  var={v:.1%}')

# ── Same experiments as Cell 6, with PLIP as the histology input ─────────────
PL_EXPS = [('P1. PLIP only',           plip_pca),
           ('P2. PLIP + Clinical',     clean_X(X_clin_N.join(plip_pca))),
           ('P3. PLIP + Genomic',      clean_X(X_genomic_N.join(plip_pca))),
           ('P4. PLIP + Clin + Gen',   clean_X(X_pca_N.join(plip_pca)))]
rows = []
for exp_name, X in PL_EXPS:
    for label in LABEL_COLS:
        y_lbl = y_4tasks[label].reindex(X.index)
        if int(y_lbl.fillna(0).sum()) < 3: continue
        for mname, model in make_models().items():
            m = run_cv(X, y_lbl, model, use_loo=True)
            if m: rows.append({'experiment':exp_name,'label':label,'model':mname,
                               'n':int(y_lbl.notna().sum()),**m})
    print(f'{exp_name} done')
plip_df = pd.DataFrame(rows)
plip_df.to_csv(f'{OUT}/plip_results.csv', index=False)

print('\n' + '='*80 + '\nPLIP — LOO, 118 patients: best model | mean of 4 models\n' + '='*80)
for label in LABEL_COLS:
    print(f'\n{label}')
    for e, _ in PL_EXPS:
        s = plip_df[(plip_df.label==label) & (plip_df.experiment==e)]
        if s.empty: continue
        b = s.loc[s.AUC.idxmax()]
        print(f'  {e:<24} best={b.AUC:.3f} ({b.model:<10}) F1={b.F1:.3f} MCC={b.MCC:.3f}  mean={s.AUC.mean():.3f}')

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
HISTO_CMAP = LinearSegmentedColormap.from_list(
    'histo', ['#000080','#0000ff','#00ffff','#ffff00','#ff0000'], N=256)

def plip_rollout(pv):
    with torch.no_grad():
        att = plip_vis(pixel_values=pv, output_attentions=True).attentions
    res = torch.eye(att[0].shape[-1], device=pv.device)
    for a in att:
        a = a.squeeze(0).mean(dim=0)
        a = a + torch.eye(a.shape[0], device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        res = a @ res
    cam = res[0, 1:]
    n = int(cam.shape[0] ** 0.5)
    cam = cam[:n*n].reshape(n, n).cpu().numpy().astype(float)
    cam = np.clip(cam, np.percentile(cam, 10), np.percentile(cam, 90))
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def show_plip_attention(pid, tag, n_tiles=4):
    paths = tumor_tiles.get(pid, [])[:n_tiles]
    if not paths: print(f'  no tumor tiles for {pid}'); return
    fig, axes = plt.subplots(3, len(paths), figsize=(4*len(paths), 11), squeeze=False)
    fig.suptitle(f'PLIP attention — {pid} ({tag})', fontsize=13, fontweight='bold')
    for c, p in enumerate(paths):
        pil = Image.open(p).convert('RGB'); img = np.array(pil)
        pv  = plip_processor(images=pil, return_tensors='pt')['pixel_values'].to(device)
        cam = np.array(Image.fromarray((plip_rollout(pv)*255).astype(np.uint8)).resize(
              (img.shape[1], img.shape[0]), Image.BICUBIC))/255.0
        rgba  = HISTO_CMAP(cam); alpha = (cam*0.55)[..., None]
        axes[0,c].imshow(img); axes[0,c].set_title(f'Tile {c+1}', fontsize=9)
        axes[1,c].imshow(cam, cmap=HISTO_CMAP, vmin=0, vmax=1)
        axes[2,c].imshow(np.clip(img/255.0*(1-alpha) + rgba[:,:,:3]*alpha, 0, 1))
        for r in range(3): axes[r,c].axis('off')
    for r, t in enumerate(['H&E','Attention','Overlay']): axes[r,0].set_ylabel(t, fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{OUT}/plip_attn_{pid}_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()

for pid in pick('label_survival', 1):              show_plip_attention(pid, 'Survival-Died')
for pid in pick('label_survival', 0):              show_plip_attention(pid, 'Survival-Alive')
for pid in pick('label_treatment_response', 1):    show_plip_attention(pid, 'Treatment-CompleteResponse')
for pid in pick('label_treatment_response', 0):    show_plip_attention(pid, 'Treatment-NonResponse')

In [21]:
import os, glob, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             balanced_accuracy_score, matthews_corrcoef)
import xgboost as xgb, lightgbm as lgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print('Saved files:', [f for f in os.listdir(OUT) if f.endswith(('.csv', '.pkl'))])

eff_pca = pd.read_csv(f'{OUT}/X_histo_eff.csv', index_col='patient_id')
vit_pca = pd.read_csv(f'{OUT}/X_histo_vit.csv', index_col='patient_id')
fc = pd.read_csv(f'{OUT}/feature_comparison.csv')
BEST_NAME  = dict(zip(fc['label'], fc['best']))
BEST_HISTO = {l: (vit_pca if BEST_NAME[l] == 'ViT' else eff_pca) for l in LABEL_COLS}

idx_N       = list(eff_pca.index)
X_clin_N    = X_clin.loc[idx_N]
X_genomic_N = X_genomic.loc[idx_N]
X_pca_N     = X_pca.loc[idx_N]
y_N         = y_4tasks.loc[idx_N]

def evaluate(y_true, y_prob):
    y_true = np.asarray(y_true); y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
    try:    auc = roc_auc_score(y_true, y_prob)
    except Exception: auc = 0.5
    return {'AUC': round(auc,3)}

def make_models():
    return {
        'SVM'       : SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
                          probability=True, random_state=42),
        'XGBoost'   : xgb.XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05,
                          subsample=0.8, scale_pos_weight=5, eval_metric='logloss', random_state=42),
        'LightGBM'  : lgb.LGBMClassifier(n_estimators=100, max_depth=2, learning_rate=0.05,
                          class_weight='balanced', random_state=42, verbose=-1),
        'ElasticNet': LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                          C=0.1, class_weight='balanced', max_iter=2000, random_state=42),
    }
print('Restored:', len(idx_N), 'patients | histo choice:', BEST_NAME)

Saved files: ['y_labels_4tasks.csv']


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/X_histo_eff.csv'

In [28]:
import glob, numpy as np, pandas as pd
from PIL import Image
from transformers import AutoConfig

slides = sorted(s for s in os.listdir(TILES_ROOT) if os.path.isdir(f'{TILES_ROOT}/{s}'))
sizes = {}
for s in slides[:40]:
    for f in glob.glob(f'{TILES_ROOT}/{s}/*.png')[:2]:
        sz = Image.open(f).size
        sizes[sz] = sizes.get(sz, 0) + 1
print('Tile sizes (sample):', sizes)
print('Slides:', len(slides))

cfg = AutoConfig.from_pretrained("vinid/plip")
print('PLIP hidden size:', cfg.vision_config.hidden_size, '| projection dim:', cfg.projection_dim)

print(y_N.apply(lambda c: pd.Series({'n': c.notna().sum(), 'pos': int(c.fillna(0).sum())})).T)
print('eff/vit index equal:', list(eff_pca.index) == list(vit_pca.index))

Tile sizes (sample): {(512, 512): 80}
Slides: 140


config.json:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

PLIP hidden size: 768 | projection dim: 512
                            n  pos
label_cancer_stage        108   32
label_node_positive       109   57
label_survival            118   25
label_recurrence          118   21
label_treatment_response  118   10
eff/vit index equal: True


In [32]:
import warnings; warnings.filterwarnings('ignore')
import lightgbm; lightgbm.register_logger  # noqa
import time
from sklearn.model_selection import LeaveOneOut, cross_val_predict
rng = np.random.default_rng(42)

def loo_prob(X, y_series, model):
    y = y_series.reindex(X.index).dropna().astype(int)
    Xv = X.loc[y.index]
    pipe = ImbPipeline([('smote', SMOTE(random_state=42, k_neighbors=min(3, int(y.sum())-1))),
                        ('scaler', StandardScaler()), ('clf', model)])
    p = cross_val_predict(pipe, Xv, y, cv=LeaveOneOut(), method='predict_proba', n_jobs=-1)[:,1]
    return y.values, p

def boot_ci(y, p, B=1000):
    a = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): a.append(roc_auc_score(y[i], p[i]))
    return np.percentile(a, [2.5, 97.5])

CASES = [
 ('Survival  Clin+Histo LightGBM', 'label_survival',
  lambda h: clean_X(X_clin_N.join(h, how='inner')), 'LightGBM'),
 ('Treatment Histo only XGBoost', 'label_treatment_response',
  lambda h: h, 'XGBoost'),
]
rows = []
for name, lab, bx, mk in CASES:
    t0 = time.time()
    X = bx(BEST_HISTO[lab])
    y, p = loo_prob(X, y_4tasks[lab], make_models()[mk])
    auc = roc_auc_score(y, p); lo, hi = boot_ci(y, p)
    print(f'{name}: AUC={auc:.3f}  95% CI {lo:.3f}-{hi:.3f}  ({time.time()-t0:.0f}s)', flush=True)
    seeds = []
    for sd in range(3):
        m = make_models()[mk]; m.set_params(random_state=sd)
        yy, pp = loo_prob(X, y_4tasks[lab], m); seeds.append(roc_auc_score(yy, pp))
    print(f'   seeds 0-2: mean={np.mean(seeds):.3f} std={np.std(seeds):.3f}', flush=True)
    enc = {}
    for en, hh in [('Eff', eff_pca), ('ViT', vit_pca)]:
        yy, pp = loo_prob(bx(hh), y_4tasks[lab], make_models()[mk]); enc[en] = round(roc_auc_score(yy, pp), 3)
    print(f'   Eff only={enc["Eff"]}  ViT only={enc["ViT"]}  ({time.time()-t0:.0f}s total)', flush=True)
    rows.append({'case': name, 'AUC': round(auc,3), 'CI95': f'{lo:.3f}-{hi:.3f}',
                 'seed_mean': round(np.mean(seeds),3), 'seed_std': round(np.std(seeds),3),
                 'Eff_only': enc['Eff'], 'ViT_only': enc['ViT']})
pd.DataFrame(rows).to_csv(f'{OUT}/headline_checks.csv', index=False)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Survival  Clin+Histo LightGBM: AUC=0.822  95% CI 0.708-0.913  (7s)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

   seeds 0-2: mean=0.822 std=0.000


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

   Eff only=0.687  ViT only=0.822  (59s total)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Treatment Histo only XGBoost: AUC=0.849  95% CI 0.737-0.933  (6s)
   seeds 0-2: mean=0.836 std=0.004
   Eff only=0.424  ViT only=0.849  (30s total)
